# DORAnet → enzyme hypotheses → DNA design

This notebook converts DORAnet-generated reaction networks into concise downstream design tables: parsed reactions, enzyme hypotheses, enzyme-candidate templates, and non-operational DNA design plans for expert review.

In [1]:
from pathlib import Path
import glob
import json
import os
import re
import textwrap
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rdkit import Chem, rdBase
from rdkit.Chem import rdmolfiles, Draw
from IPython.display import display
import time
import requests
from io import StringIO
import ast
import sys
import yaml
from tqdm.auto import tqdm
tqdm.pandas()

from DORA_XGB import DORA_XGB
by_desc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_descending_MW')
by_asc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_ascending_MW')
add_concat_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_concat')
add_subtract_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_subtract')

from Bio.Data import CodonTable

from dnachisel import (
    DnaOptimizationProblem,
    EnforceTranslation,
    EnforceGCContent,
    AvoidPattern,
    MaximizeCAI,
)

import hashlib

/users/sghosh6/.conda/envs/doranet_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/users/sghosh6/.conda/envs/doranet_env/lib/python3.10/site-packages/xgboost/compat.py:105: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
resultsDir = os.path.join(dataDir, 'Results/combinedEbolaVirus_bestMACAW_allDB_generative/')
DORANETmoleculesDataDir = os.path.join(resultsDir, 'DORAnet/InoAdeGuoXan/PATHWAY_reconstruction/')
#DORANETmoleculesDataDir = os.path.join(resultsDir, 'DORAnet/InoAdeGuoXan/PATHWAY_reconstruction_run2/')
DNADesignResultsDir = os.path.join(resultsDir, 'DNA_design/')
os.makedirs(DNADesignResultsDir, exist_ok=True)

print("Reading DORAnet results from:", DORANETmoleculesDataDir)
print("DNA design results will be saved at:", DNADesignResultsDir)

Reading DORAnet results from: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/PATHWAY_reconstruction/
DNA design results will be saved at: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/


## 0. Read list of target DORAnet compounds

In [3]:
targetCompound_DF = pd.read_csv(resultsDir + "/combinedEbola_allDORAnetGenerated_Antivirals_wARTprediction_top20.csv")
targetCompound_DF

,Rank,Canonical_SMILES,pPotency_prediction,pPotency_std,IC50 (µM)
0,1,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,6.156,0.476241,0.697981
1,2,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,5.989,0.473052,1.025573
2,3,Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O...,5.907,0.473610,1.238912
3,4,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H](O)[C@H](O)...,5.825,0.472941,1.495337
4,5,CC(=O)OP(=O)(O)OC(C)=O,5.758,0.479591,1.747229
5,6,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,5.729,0.473567,1.865382
6,7,CC(=O)OP(=O)(O)OP(=O)(O)O,5.693,0.478908,2.026785
7,8,Nc1ncnc2c1ncn2[C@@H]1OC(C(O)O)=C[C@H]1O,5.692,0.473252,2.030622
8,9,CC(Cl)[C@H]1O[C@@H](n2cnc3c(NC=O)ncnc32)[C@H](...,5.683,0.471067,2.074770
9,10,O=C(O)CC(=O)OP(=O)(O)O,5.679,0.476017,2.094051


In [4]:
DORANETmoleculesDataDir = Path(DORANETmoleculesDataDir)

def canonicalizeSmiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol, canonical=True) if mol else None

def canonicalizeSmilesCollection(x):
    if not isinstance(x, (set, list, tuple)):
        return []
    out = []
    for s in x:
        c = canonicalizeSmiles(s)
        if c is not None:
            out.append(c)
    return sorted(set(out))

def parseLiteralFromScript(text, varName):
    # captures lines like: varName = {...} / [...] / "..."
    m = re.search(rf"^\s*{re.escape(varName)}\s*=\s*(.+)\s*$", text, flags=re.MULTILINE)
    if not m:
        return None
    raw = m.group(1).strip()
    try:
        return ast.literal_eval(raw)
    except Exception:
        return None

def parseHelpersFromScript(text):
    # Supports YAML-like line: helpers: ["O", ...]
    m = re.search(r"^\s*helpers\s*:\s*(.+)\s*$", text, flags=re.MULTILINE)
    if not m:
        # fallback in case file uses python assignment: helpers = [...]
        return parseLiteralFromScript(text, "helpers")
    raw = m.group(1).strip()
    try:
        return ast.literal_eval(raw)
    except Exception:
        return None

allReproFiles = sorted(
    DORANETmoleculesDataDir.rglob("reproDoranetJob.py"),
    key=lambda p: str(p).lower()
)

records = []
for reproPath in allReproFiles:
    txt = reproPath.read_text(encoding="utf-8", errors="ignore")

    startersRaw = parseLiteralFromScript(txt, "starters")
    targetRaw = parseLiteralFromScript(txt, "target")
    maxAtomsRaw = parseLiteralFromScript(txt, "maxAtoms")
    generationsRaw = parseLiteralFromScript(txt, "generations")
    helpersRaw = parseHelpersFromScript(txt)

    records.append({
        "dirName": reproPath.parent.name,
        "starters": canonicalizeSmilesCollection(startersRaw),
        "target": canonicalizeSmilesCollection(targetRaw),
        "helpers": canonicalizeSmilesCollection(helpersRaw),   # NEW
        "maxAtoms": maxAtomsRaw if isinstance(maxAtomsRaw, dict) else None,
        "generations": int(generationsRaw) if generationsRaw is not None else None,
    })

targetCompoundDir_DF = pd.DataFrame(
    records,
    columns=["dirName", "starters", "target", "helpers", "maxAtoms", "generations"]  # NEW
)

print(f"Total DORAnet configuration files: {len(targetCompoundDir_DF)}")

# join key from first target smiles
targetCompoundDir_DF = targetCompoundDir_DF.copy()
targetCompoundDir_DF["targetSmiles"] = targetCompoundDir_DF["target"].apply(
    lambda x: x[0] if isinstance(x, (list, tuple)) and len(x) > 0 else None
)

colsToAdd = ["Canonical_SMILES", "pPotency_prediction", "pPotency_std", "IC50 (µM)"]
targetCompoundDir_DF = targetCompoundDir_DF.merge(
    targetCompound_DF[colsToAdd].drop_duplicates(subset=["Canonical_SMILES"]),
    left_on="targetSmiles",
    right_on="Canonical_SMILES",
    how="left"
).drop(columns=["Canonical_SMILES"])

targetCompoundDir_DF = targetCompoundDir_DF.drop(columns=["targetSmiles"])
targetCompoundDir_DF = targetCompoundDir_DF.rename(columns={
    "starters": "starters_Canonical_SMILES",
    "target": "target_Canonical_SMILES",
    "helpers": "helpers_Canonical_SMILES",   
})

for c in ["starters_Canonical_SMILES", "target_Canonical_SMILES"]:
    targetCompoundDir_DF[c] = targetCompoundDir_DF[c].apply(
        lambda x: ".".join(x) if isinstance(x, (list, tuple)) else ("" if pd.isna(x) else str(x))
    )

# Keep helpers as separate entries joined by comma (not dot)
targetCompoundDir_DF["helpers_Canonical_SMILES"] = targetCompoundDir_DF["helpers_Canonical_SMILES"].apply(
    lambda x: ", ".join(x) if isinstance(x, (list, tuple)) else ("" if pd.isna(x) else str(x))
)

targetCompoundDir_DF = targetCompoundDir_DF.sort_values(
    by="pPotency_prediction", ascending=False
).reset_index(drop=True)

# safer path handling
outputPath = Path(DNADesignResultsDir) / "targetCompoundDir_DF.csv"
targetCompoundDir_DF.to_csv(outputPath, index=False)
targetCompoundDir_DF

Total DORAnet configuration files: 20


,dirName,starters_Canonical_SMILES,target_Canonical_SMILES,helpers_Canonical_SMILES,maxAtoms,generations,pPotency_prediction,pPotency_std,IC50 (µM)
0,high_pPotency_molecule_pathway1,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,6.156,0.476241,0.697981
1,high_pPotency_molecule_pathway2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.989,0.473052,1.025573
2,high_pPotency_molecule_pathway3,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.907,0.473610,1.238912
3,high_pPotency_molecule_pathway4,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H](O)[C@H](O)...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.825,0.472941,1.495337
4,high_pPotency_molecule_pathway5,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(=O)OP(=O)(O)OC(C)=O,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.758,0.479591,1.747229
5,high_pPotency_molecule_pathway6,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.729,0.473567,1.865382
6,high_pPotency_molecule_pathway7,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(=O)OP(=O)(O)OP(=O)(O)O,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.693,0.478908,2.026785
7,high_pPotency_molecule_pathway8,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,Nc1ncnc2c1ncn2[C@@H]1OC(C(O)O)=C[C@H]1O,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.692,0.473252,2.030622
8,high_pPotency_molecule_pathway9,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(Cl)[C@H]1O[C@@H](n2cnc3c(NC=O)ncnc32)[C@H](...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.683,0.471067,2.074770
9,high_pPotency_molecule_pathway10,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,O=C(O)CC(=O)OP(=O)(O)O,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.679,0.476017,2.094051


## 1. Read DORAnet reaction network files

In [5]:
DORANETmoleculesDataDir = Path(DORANETmoleculesDataDir)  

def extractFirstInt(text):
    match = re.search(r"\d+", text)
    return int(match.group(0)) if match else -1

# Find all JSON files anywhere under root
allJsonPaths = sorted(DORANETmoleculesDataDir.rglob("*.json"), key=lambda p: str(p).lower())

fileInfoList = []
for jsonPath in allJsonPaths:
    parentDir = jsonPath.parent
    fileInfoList.append({
        "dirName": parentDir.name,
        "dirPath": str(parentDir),
        "jsonName": jsonPath.name,
        "jsonPath": str(jsonPath),
        "folderNum": extractFirstInt(parentDir.name),  # optional helper field
    })

print(f"Root directory        : {DORANETmoleculesDataDir}")
print(f"JSON files found      : {len(fileInfoList)}")
print(f"Unique folders w/JSON : {len({x['dirPath'] for x in fileInfoList})}")

Root directory        : /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/PATHWAY_reconstruction
JSON files found      : 20
Unique folders w/JSON : 20


## 2.1 Parse DORAnet reaction pathways

In [6]:
def splitMoleculeString(moleculeString):
    return [mol for mol in str(moleculeString).split(".") if mol]

def parseDoranetReaction(rxnString, fileInfo):
    parts = str(rxnString).split(">")
    if len(parts) != 4:
        raise ValueError(f"Expected 4 fields separated by '>'; found {len(parts)}")

    reactants, ruleName, metaBlock, products = parts
    metaParts = (metaBlock.split("$") + [None, None, None, None])[:4]
    thermo, reactantStoich, productStoich, reactionType = metaParts

    return {
        "reactants": reactants,
        "products": products,
        "reactionString": f"{reactants} >> {products}",
        "ruleName": ruleName,
        "thermo": thermo,
        "reactantStoich": reactantStoich,
        "productStoich": productStoich,
        "reactionType": reactionType,
        "numReactantMolecules": len(splitMoleculeString(reactants)),
        "numProductMolecules": len(splitMoleculeString(products)),
        "sourceFolderNum": fileInfo.get("folderNum", -1),
        "sourceDirectory": fileInfo.get("dirName", ""),
        "sourceDirectoryPath": fileInfo.get("dirPath", ""),
        "sourceJsonName": fileInfo.get("jsonName", ""),
        "sourceJsonPath": fileInfo.get("jsonPath", ""),
    }

reactionRecords = []
jsonReadErrors = []
uniqueReactantMolecules = set()
uniqueProductMolecules = set()

for fileInfo in tqdm(fileInfoList, desc="Reading DORAnet JSON"):
    try:
        with open(fileInfo["jsonPath"], "r", encoding="utf-8") as f:
            reactionList = json.load(f)

        # handle both list and dict payloads
        if isinstance(reactionList, dict):
            # try common key names; otherwise fail clearly
            for k in ["reactions", "reactionList", "data"]:
                if k in reactionList and isinstance(reactionList[k], list):
                    reactionList = reactionList[k]
                    break
            else:
                raise ValueError("JSON is dict but no reaction list key found")

        if not isinstance(reactionList, list):
            raise ValueError(f"Expected list of reaction strings, got {type(reactionList)}")

        for rxnString in reactionList:
            record = parseDoranetReaction(rxnString, fileInfo)
            reactionRecords.append(record)
            uniqueReactantMolecules.update(splitMoleculeString(record["reactants"]))
            uniqueProductMolecules.update(splitMoleculeString(record["products"]))

    except Exception as exc:
        jsonReadErrors.append({**fileInfo, "error": str(exc)})

if not reactionRecords:
    raise RuntimeError("No reactions loaded. Check JSON discovery and reaction format.")

reactionDF = pd.DataFrame(reactionRecords).reset_index(drop=True)

# safer path handling
outputPath = Path(DNADesignResultsDir) / "reactionDF.csv"
reactionDF.to_csv(outputPath, index=False)

print(f"Reactions loaded     : {len(reactionDF):,}")
print(f"Failed JSON files    : {len(jsonReadErrors):,}")
print(f"Unique reactant mols : {len(uniqueReactantMolecules):,}")
print(f"Unique product mols  : {len(uniqueProductMolecules):,}")
print(f"Saved: {outputPath}")

reactionDF.head()

Reading DORAnet JSON: 100%|████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 22.69it/s]


Reactions loaded     : 176,400
Failed JSON files    : 0
Unique reactant mols : 283
Unique product mols  : 4,730
Saved: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/reactionDF.csv


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,sourceFolderNum,sourceDirectory,sourceDirectoryPath,sourceJsonName,sourceJsonPath
0,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](C=O)O[C@H]1n1cnc...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
1,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,COC[C@]1(C)O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,O=c1[nH]c(C(O)O)nc2c1ncn2[C@@H]1O[C@@H]2O[C@H]...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,rule0028_50,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
3,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,rule0001_87,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
4,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,COC(C)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...


In [7]:
print(reactionDF['reactionString'].iloc[0])
print(reactionDF['reactionString'].iloc[1])
print(reactionDF['reactionString'].iloc[2])

#print(reactionDF['reactionString'].iloc[4])

CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2c(=O)[nH]cnc21.NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)(O)O)[C@@H]3O)[C@@H](O)[C@H]2O)C=CC1.O=O >> CO[C@@H]1[C@H](OC(C)=O)[C@@H](C=O)O[C@H]1n1cnc2c(=O)[nH]cnc21.NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)(O)O)[C@@H]3O)[C@@H](O)[C@H]2O)c1.O.O
COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)[C@@H]1OC(C)=O >> COC[C@]1(C)O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)[C@@H]1OC=O
O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[C@H]1O >> O=c1[nH]c(C(O)O)nc2c1ncn2[C@@H]1O[C@@H]2O[C@H]2[C@H]1O


## 2.2 Read the `reconstructed` pathways metadata for all `DORAnet` reactions

- go to the directory where pathway reconstruction DORAnet results are saved
- run this: `python pathwayReconstruction.py config.yaml`
- this will generate the csv file (`doranet_chain_reconstructed_pathway_steps_by_depth.csv`) with reconstructed pathways
- read the csv file for downstream processing

In [13]:
# -------------------------------------------------------------------
# settings
# -------------------------------------------------------------------
pathwayReconstructionConfigPath = Path(DNADesignResultsDir + "config.yaml")
keepOnlySuccessfulRoutes = True
maxRoutesPerFolderDepth = None  
saveOutput = True


# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------
def splitMoleculeString(moleculeString):
    if pd.isna(moleculeString):
        return []
    return [mol for mol in str(moleculeString).split(".") if mol]


def extractFolderNum(dirName):
    match = re.search(r"(\d+)$", str(dirName))
    return int(match.group(1)) if match else -1


def parseDoranetReaction(rxnString, fileInfo):
    """
    Parse original DORAnet reaction string:

        reactants > ruleName > thermo$reactantStoich$productStoich$reactionType > products

    Returns a reactionDF-like record.
    """
    parts = str(rxnString).split(">")

    if len(parts) != 4:
        raise ValueError(f"Expected 4 fields separated by '>'; found {len(parts)}")

    reactants, ruleName, metaBlock, products = parts

    metaParts = (metaBlock.split("$") + [None, None, None, None])[:4]
    thermo, reactantStoich, productStoich, reactionType = metaParts

    return {
        "reactants": reactants,
        "products": products,
        "reactionString": f"{reactants} >> {products}",
        "ruleName": ruleName,
        "thermo": thermo,
        "reactantStoich": reactantStoich,
        "productStoich": productStoich,
        "reactionType": reactionType,
        "numReactantMolecules": len(splitMoleculeString(reactants)),
        "numProductMolecules": len(splitMoleculeString(products)),
        "sourceFolderNum": fileInfo.get("folderNum", -1),
        "sourceDirectory": fileInfo.get("dirName", ""),
        "sourceDirectoryPath": fileInfo.get("dirPath", ""),
        "sourceJsonName": fileInfo.get("jsonName", ""),
        "sourceJsonPath": fileInfo.get("jsonPath", ""),
    }


def parseCleanReactionSmiles(reactionSmiles, fileInfo, row):
    """
    Fallback parser for clean reaction SMILES:

        reactants >> products

    Used only if rawReactionString is missing or malformed.
    """
    if pd.isna(reactionSmiles) or ">>" not in str(reactionSmiles):
        raise ValueError("No valid reactionSMILES fallback found")

    reactants, products = str(reactionSmiles).split(">>", 1)

    return {
        "reactants": reactants,
        "products": products,
        "reactionString": f"{reactants} >> {products}",
        "ruleName": row.get("reactionName", ""),
        "thermo": row.get("dH", ""),
        "reactantStoich": None,
        "productStoich": None,
        "reactionType": row.get("reactionType", ""),
        "numReactantMolecules": len(splitMoleculeString(reactants)),
        "numProductMolecules": len(splitMoleculeString(products)),
        "sourceFolderNum": fileInfo.get("folderNum", -1),
        "sourceDirectory": fileInfo.get("dirName", ""),
        "sourceDirectoryPath": fileInfo.get("dirPath", ""),
        "sourceJsonName": fileInfo.get("jsonName", ""),
        "sourceJsonPath": fileInfo.get("jsonPath", ""),
    }


def resolveOutputDirFromConfig(pathwayReconstructionConfigPath):
    with open(pathwayReconstructionConfigPath, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}

    outputDir = Path(
        cfg.get("paths", {}).get("output_dir", "pathway_reconstruction_results")
    ).expanduser()

    if not outputDir.is_absolute():
        outputDir = Path.cwd() / outputDir

    return outputDir.resolve(), cfg


# -------------------------------------------------------------------
# Locate reconstructed pathway output files
# -------------------------------------------------------------------
outputDir, cfg = resolveOutputDirFromConfig(pathwayReconstructionConfigPath)

stepsPath = Path(DNADesignResultsDir +  "pathway_reconstruction_results/doranet_chain_reconstructed_pathway_steps_by_depth.csv")
routesPath = Path(DNADesignResultsDir +  "pathway_reconstruction_results/doranet_chain_reconstructed_pathways_by_depth.csv")
summaryPath = Path(DNADesignResultsDir +  "pathway_reconstruction_results/doranet_chain_pathway_summary_by_depth.csv")

if not stepsPath.exists():
    raise FileNotFoundError(
        f"Could not find reconstructed pathway steps file:\n{stepsPath}\n\n"
        "Run pathwayReconstruction.py first."
    )

stepsDF = pd.read_csv(stepsPath)

if stepsDF.empty:
    raise RuntimeError(
        f"The reconstructed pathway step file is empty:\n{stepsPath}\n\n"
        "No reconstructed pathways were found."
    )


# -------------------------------------------------------------------
# keep only routes present in route-level file
# -------------------------------------------------------------------
if keepOnlySuccessfulRoutes and routesPath.exists():
    routesDF = pd.read_csv(routesPath)
    validRouteIds = set(routesDF["routeId"].dropna().astype(str))
    stepsDF = stepsDF[stepsDF["routeId"].astype(str).isin(validRouteIds)].copy()


# -------------------------------------------------------------------
# limit number of routes per folder/depth
# -------------------------------------------------------------------
if maxRoutesPerFolderDepth is not None:
    routeSelectorDF = (
        stepsDF[["dirName", "searchDepthUsed", "routeId"]]
        .drop_duplicates()
        .sort_values(["dirName", "searchDepthUsed", "routeId"])
        .groupby(["dirName", "searchDepthUsed"], as_index=False)
        .head(maxRoutesPerFolderDepth)
    )

    keepRouteIds = set(routeSelectorDF["routeId"].astype(str))
    stepsDF = stepsDF[stepsDF["routeId"].astype(str).isin(keepRouteIds)].copy()


# -------------------------------------------------------------------
# Parse reconstructed route steps into reactionDF-like records
# -------------------------------------------------------------------
reconstructedRecords = []
parseErrors = []

for _, row in tqdm(stepsDF.iterrows(), total=len(stepsDF), desc="Parsing reconstructed pathways"):
    dirName = row.get("dirName", "")
    jobName = row.get("jobName", "")

    fileInfo = {
        "folderNum": extractFolderNum(dirName),
        "dirName": dirName,
        "dirPath": str(Path(cfg.get("paths", {}).get("doranet_jobs_dir", "")) / str(dirName)),
        "jsonName": f"{jobName}_network_pretreated.json" if jobName else "",
        "jsonPath": str(
            Path(cfg.get("paths", {}).get("doranet_jobs_dir", ""))
            / str(dirName)
            / f"{jobName}_network_pretreated.json"
        ) if jobName else "",
    }

    try:
        rawReactionString = row.get("rawReactionString", None)

        if pd.notna(rawReactionString) and len(str(rawReactionString).split(">")) == 4:
            record = parseDoranetReaction(rawReactionString, fileInfo)
        else:
            record = parseCleanReactionSmiles(row.get("reactionSMILES", None), fileInfo, row)

        # Add reconstructed-pathway metadata
        record.update({
            "dirName": dirName,
            "jobName": jobName,
            "routeId": row.get("routeId", ""),
            "searchDepthUsed": row.get("searchDepthUsed", None),
            "step": row.get("step", None),
            "numSteps": row.get("numSteps", None),
            "rxnId": row.get("rxnId", None),

            # Chain-connection metadata
            "connectionType": row.get("connectionType", ""),
            "connectionMolecules": row.get("connectionMolecules", ""),
            "sideReactantsAllowedAsCofactors": row.get("sideReactantsAllowedAsCofactors", ""),
            "sideReactantsListedAsHelpers": row.get("sideReactantsListedAsHelpers", ""),
            "sideReactantsNotListedAsHelpers": row.get("sideReactantsNotListedAsHelpers", ""),
            "starterInReactants": row.get("starterInReactants", None),
            "targetInProducts": row.get("targetInProducts", None),

            # Useful downstream identifiers
            "routeStepId": f"{row.get('routeId', '')}_step_{int(row.get('step', 0)):02d}"
            if pd.notna(row.get("step", None)) else "",
            "isReconstructedPathwayStep": True,
        })

        reconstructedRecords.append(record)

    except Exception as exc:
        parseErrors.append({
            "dirName": dirName,
            "jobName": jobName,
            "routeId": row.get("routeId", ""),
            "step": row.get("step", None),
            "reactionSMILES": row.get("reactionSMILES", ""),
            "rawReactionString": row.get("rawReactionString", ""),
            "error": str(exc),
        })


if not reconstructedRecords:
    raise RuntimeError("No reconstructed pathway records could be parsed.")


reconstructedPathwaysDF = (
    pd.DataFrame(reconstructedRecords)
    .sort_values(["sourceFolderNum", "searchDepthUsed", "routeId", "step"])
    .reset_index(drop=True)
)

parseErrorsDF = pd.DataFrame(parseErrors)


# -------------------------------------------------------------------
# Add route-level summary columns to every step
# -------------------------------------------------------------------
if routesPath.exists():
    routeColsToMerge = [
        "routeId",
        "multiStepReactionSMILES",
        "reactionNames",
        "starterSMILES",
        "targetSMILES",
        "connectionMoleculesByStep",
    ]

    routesDF = pd.read_csv(routesPath)
    routeColsToMerge = [c for c in routeColsToMerge if c in routesDF.columns]

    reconstructedPathwaysDF = reconstructedPathwaysDF.merge(
        routesDF[routeColsToMerge].drop_duplicates("routeId"),
        on="routeId",
        how="left",
    )


# -------------------------------------------------------------------
# Save for downstream processing
# -------------------------------------------------------------------
reconstructedPathwaysDF = reconstructedPathwaysDF.rename(columns={'multiStepReactionSMILES' :'reconstructedPathwayString'})
if saveOutput:
    reconstructedPathwaysDF.to_csv(DNADesignResultsDir + "reconstructedPathwaysDF.csv", index=False)
    parseErrorsDF.to_csv(DNADesignResultsDir + "reconstructedPathwaysDF_parseErrors.csv", index=False)

    print(f"Saved reconstructedPathway metadata to {DNADesignResultsDir}")


print(f"Reconstructed pathway steps : {len(reconstructedPathwaysDF):,}")
print(f"Unique routes               : {reconstructedPathwaysDF['routeId'].nunique():,}")
print(f"Unique folders              : {reconstructedPathwaysDF['dirName'].nunique():,}")
print(f"Parse errors                : {len(parseErrorsDF):,}")

reconstructedPathwaysDF.head() 

Parsing reconstructed pathways: 100%|█████████████████████████████████████████████████████████████| 12730/12730 [00:01<00:00, 11272.96it/s]


Saved reconstructedPathway metadata to /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/
Reconstructed pathway steps : 12,730
Unique routes               : 4,489
Unique folders              : 7
Parse errors                : 0


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,sideReactantsNotListedAsHelpers,starterInReactants,targetInProducts,routeStepId,isReconstructedPathwayStep,reconstructedPathwayString,reactionNames,starterSMILES,targetSMILES,connectionMoleculesByStep
0,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,True,False,high_pPotency_molecule_pathway1_d3_route_00000...,True,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,rule0073_5 | rule0007_203 | rule0061_4,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O
1,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,rule0007_203,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,COC1C(COC(C)=O)OC(n2cnc3c(=O)[nH]cnc32)C1O,False,False,high_pPotency_molecule_pathway1_d3_route_00000...,True,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,rule0073_5 | rule0007_203 | rule0061_4,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,CC(C)(COP(=O)(O)OP(=O)(O)OCC1OC(n2cnc3c(N)ncnc...,False,True,high_pPotency_molecule_pathway1_d3_route_00000...,True,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,rule0073_5 | rule0007_203 | rule0061_4,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O
3,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)...,O.O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](OP...,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)...,rule0017_27,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O=P(O)(O)O,True,False,high_pPotency_molecule_pathway1_d3_route_00000...,True,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O>>...,rule0017_27 | rule0007_203 | rule0061_4,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O
4,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,rule0007_203,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,COC1C(COC(C)=O)OC(n2cnc3c(=O)[nH]cnc32)C1O,False,False,high_pPotency_molecule_pathway1_d3_route_00000...,True,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O>>...,rule0017_27 | rule0007_203 | rule0061_4,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O


### Print `reconstructed` pathways

In [14]:
print(reconstructedPathwaysDF['reconstructedPathwayString'].iloc[0])

NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4cnc5c(N)ncnc54)C(OP(=O)(O)O)C3O)C(O)C2O)C=CC1.O=O.O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O>>NC(=O)c1ccc[n+](C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4cnc5c(N)ncnc54)C(OP(=O)(O)O)C3O)C(O)C2O)c1.O.O=CC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O  ||  COC1C(COC(C)=O)OC(n2cnc3c(=O)[nH]cnc32)C1O.O>>CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O.CO  ||  CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O.CC(C)(COP(=O)(O)OP(=O)(O)OCC1OC(n2cnc3c(N)ncnc32)C(O)C1OP(=O)(O)O)C(O)C(=O)NCCC(=O)NCCS>>CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP(=O)(O)OCC1OC(n2cnc3c(N)ncnc32)C(O)C1OP(=O)(O)O.O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O


### Find unique `starter --> target` from reconstructed pathways

In [21]:
# Count complete starter --> target reconstructed pathways per directory
to_bool = lambda s: s.astype(str).str.lower().isin(["true", "1", "yes"])

first = reconstructedPathwaysDF[reconstructedPathwaysDF["step"].eq(1)]
final = reconstructedPathwaysDF[reconstructedPathwaysDF["step"].eq(reconstructedPathwaysDF["numSteps"])]

complete_ids = set(first.loc[to_bool(first["starterInReactants"]), "routeId"].astype(str)) & \
               set(final.loc[to_bool(final["targetInProducts"]), "routeId"].astype(str))

completeRouteLevelDF = (
    reconstructedPathwaysDF[reconstructedPathwaysDF["routeId"].astype(str).isin(complete_ids)]
    .sort_values(["dirName", "routeId", "step"])
    .groupby("routeId", as_index=False)
    .agg(
        dirName=("dirName", "first"),
        jobName=("jobName", "first"),
        sourceFolderNum=("sourceFolderNum", "first"),
        numSteps=("numSteps", "first"),
        searchDepthUsed=("searchDepthUsed", "first"),
        starterSMILES=("starterSMILES", "first"),
        targetSMILES=("targetSMILES", "first"),
        reconstructedPathwayString=("reconstructedPathwayString", "first"),
    )
)

starterTargetPathwayCountsDF = (
    completeRouteLevelDF
    .pivot_table(
        index=["dirName", "jobName", "starterSMILES", "targetSMILES"],
        columns="numSteps",
        values="routeId",
        aggfunc="nunique",
        fill_value=0,
    )
    .rename(columns={1: "numOneStepPathways", 2: "numTwoStepPathways", 3: "numThreeStepPathways"})
    .reset_index()
)

for c in ["numOneStepPathways", "numTwoStepPathways", "numThreeStepPathways"]:
    if c not in starterTargetPathwayCountsDF.columns:
        starterTargetPathwayCountsDF[c] = 0

starterTargetPathwayCountsDF["totalStarterToTargetPathways"] = (
    starterTargetPathwayCountsDF["numOneStepPathways"] +
    starterTargetPathwayCountsDF["numTwoStepPathways"] +
    starterTargetPathwayCountsDF["numThreeStepPathways"]
)

starterTargetPathwayCountsDF = starterTargetPathwayCountsDF.sort_values(
    "totalStarterToTargetPathways", ascending=False
).reset_index(drop=True)

print(f"Total reaction pathways : {len(reconstructedPathwaysDF):,}")
print(f"Complete starter --> target pathways: {len(completeRouteLevelDF):,}")
print(f"Directories reached starter --> target pathways: {starterTargetPathwayCountsDF['dirName'].nunique():,}")

starterTargetPathwayCountsDF

Total reaction pathways : 12,730
Complete starter --> target pathways: 4,489
Directories reached starter --> target pathways: 7


numSteps,dirName,jobName,starterSMILES,targetSMILES,numOneStepPathways,numTwoStepPathways,numThreeStepPathways,totalStarterToTargetPathways
0,high_pPotency_molecule_pathway6,high_pPotency_molecule_pathway6,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,3,670,797,1470
1,high_pPotency_molecule_pathway2,high_pPotency_molecule_pathway2,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(C)(COP(=O)(O)OP(=O)(O)OCC1OC(n2cnc3c(N)ncnc...,3,27,1018,1048
2,high_pPotency_molecule_pathway16,high_pPotency_molecule_pathway16,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,O=C(O)CC(=O)O,0,21,846,867
3,high_pPotency_molecule_pathway19,high_pPotency_molecule_pathway19,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)CC(=O)O,0,0,508,508
4,high_pPotency_molecule_pathway1,high_pPotency_molecule_pathway1,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,0,0,315,315
5,high_pPotency_molecule_pathway18,high_pPotency_molecule_pathway18,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,NC(=O)c1ccc[n+](C2OC(COP(=O)(O)OP(=O)(O)OCC3OC...,1,5,274,280
6,high_pPotency_molecule_pathway12,high_pPotency_molecule_pathway12,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,O=C1N=C2NC=Nc3c2ncn3C2OC1C(O)C2O,0,0,1,1


### Keep only enzymatic DORAnet reactions

In [23]:
enzymaticReactionDF = reconstructedPathwaysDF[
    reactionDF["reactionType"].astype(str).str.lower().str.contains(
        "enzyme|enzymatic|bio|biological",
        na=False
    )
].copy()

print(f"Total reactions: {len(reconstructedPathwaysDF):,}")
print(f"Likely enzymatic reactions: {len(enzymaticReactionDF):,}")

enzymaticReactionDF.head()

Total reactions: 12,730
Likely enzymatic reactions: 12,730


/tmp/ipykernel_2079939/1451588661.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  enzymaticReactionDF = reconstructedPathwaysDF[


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,sideReactantsNotListedAsHelpers,starterInReactants,targetInProducts,routeStepId,isReconstructedPathwayStep,reconstructedPathwayString,reactionNames,starterSMILES,targetSMILES,connectionMoleculesByStep
0,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,True,False,high_pPotency_molecule_pathway1_d3_route_00000...,True,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,rule0073_5 | rule0007_203 | rule0061_4,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O
1,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,rule0007_203,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,COC1C(COC(C)=O)OC(n2cnc3c(=O)[nH]cnc32)C1O,False,False,high_pPotency_molecule_pathway1_d3_route_00000...,True,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,rule0073_5 | rule0007_203 | rule0061_4,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,CC(C)(COP(=O)(O)OP(=O)(O)OCC1OC(n2cnc3c(N)ncnc...,False,True,high_pPotency_molecule_pathway1_d3_route_00000...,True,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,rule0073_5 | rule0007_203 | rule0061_4,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O
3,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)...,O.O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](OP...,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)...,rule0017_27,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O=P(O)(O)O,True,False,high_pPotency_molecule_pathway1_d3_route_00000...,True,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O>>...,rule0017_27 | rule0007_203 | rule0061_4,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O
4,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,rule0007_203,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,COC1C(COC(C)=O)OC(n2cnc3c(=O)[nH]cnc32)C1O,False,False,high_pPotency_molecule_pathway1_d3_route_00000...,True,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O>>...,rule0017_27 | rule0007_203 | rule0061_4,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O


## 2. Use `DORA-XGB` to get feasibility score

In [24]:
reactionDF_DORAXGB = enzymaticReactionDF.copy()
#reactionDF_DORAXGB = enzymaticReactionDF.head(50).copy()

# Clean reaction string for model input
reactionDF_DORAXGB["rxn_str"] = (
    reactionDF_DORAXGB["reactionString"]
    .astype(str)
    .str.replace(" ", "", regex=False)
)

def getFeasibilityScoresAndLabels(rxnStr):
    return pd.Series({
        "feasibilityScore_rule1": by_desc_MW_model.predict_proba(rxnStr),
        "feasibilityLabel_rule1": by_desc_MW_model.predict_label(rxnStr),

        "feasibilityScore_rule2": by_asc_MW_model.predict_proba(rxnStr),
        "feasibilityLabel_rule2": by_asc_MW_model.predict_label(rxnStr),

        "feasibilityScore_rule3": add_concat_model.predict_proba(rxnStr),
        "feasibilityLabel_rule3": add_concat_model.predict_label(rxnStr),

        "feasibilityScore_rule4": add_subtract_model.predict_proba(rxnStr),
        "feasibilityLabel_rule4": add_subtract_model.predict_label(rxnStr),
    })


reactionDF_DORAXGB["rxn_str"] = (
    reactionDF_DORAXGB["reactionString"]
    .astype(str)
    .str.replace(" ", "", regex=False)
)

reactionDF_DORAXGB[
    [
        "feasibilityScore_rule1",
        "feasibilityLabel_rule1",
        "feasibilityScore_rule2",
        "feasibilityLabel_rule2",
        "feasibilityScore_rule3",
        "feasibilityLabel_rule3",
        "feasibilityScore_rule4",
        "feasibilityLabel_rule4",
    ]
] = reactionDF_DORAXGB["rxn_str"].apply(getFeasibilityScoresAndLabels)

reactionDF_DORAXGB.head()

,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,connectionMoleculesByStep,rxn_str,feasibilityScore_rule1,feasibilityLabel_rule1,feasibilityScore_rule2,feasibilityLabel_rule2,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4
0,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,0.002562,0.0,0.738403,1.0,0.627660,1.0,0.740460,1.0
1,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,rule0007_203,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,0.029845,0.0,0.343784,0.0,0.247226,0.0,0.590581,0.0
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,0.297114,0.0,0.995089,1.0,0.924887,1.0,0.990581,1.0
3,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)...,O.O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](OP...,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)...,rule0017_27,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)...,0.001216,0.0,0.034483,0.0,0.027119,0.0,0.511955,0.0
4,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,rule0007_203,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,0.029845,0.0,0.343784,0.0,0.247226,0.0,0.590581,0.0


### Keep only `high feasible` reactions

In [27]:
# Choose which feasibility label to use
feasibilityLabel = "feasibilityLabel_rule2"   # <- change to rule2/rule3/rule4 as needed
targetValue = 1                               # <- change if you want a different label value

# Optional safety check
validLabels = [
    "feasibilityLabel_rule1",
    "feasibilityLabel_rule2",
    "feasibilityLabel_rule3",
    "feasibilityLabel_rule4",
]
if feasibilityLabel not in validLabels:
    raise ValueError(f"Invalid feasibilityLabel: {feasibilityLabel}. Choose from {validLabels}")

if feasibilityLabel not in reactionDF_DORAXGB.columns:
    raise KeyError(f"Column '{feasibilityLabel}' not found in reactionDF_DORAXGB")

In [28]:
reactionDF_DORAXGB_highFeasibility = (
    reactionDF_DORAXGB[
        (reactionDF_DORAXGB[feasibilityLabel] == targetValue)
        & (reactionDF_DORAXGB[feasibilityLabel].notna())
    ]
    .sort_values(by=feasibilityLabel, ascending=False)
    .reset_index(drop=True)
)

uniqueReactantStrings_high = set(reactionDF_DORAXGB_highFeasibility["reactants"])
uniqueProductStrings_high = set(reactionDF_DORAXGB_highFeasibility["products"])

uniqueReactantMolecules_high = {
    mol.strip()
    for reactants in reactionDF_DORAXGB_highFeasibility["reactants"]
    for mol in str(reactants).split(".")
    if mol.strip()
}

uniqueProductMolecules_high = {
    mol.strip()
    for products in reactionDF_DORAXGB_highFeasibility["products"]
    for mol in str(products).split(".")
    if mol.strip()
}

print(f"Feasibility label used: {feasibilityLabel} == {targetValue}")
print(f"Number of high-feasibility reactions: {len(reactionDF_DORAXGB_highFeasibility)}")
print(f"Number of unique reactant strings: {len(uniqueReactantStrings_high)}")
print(f"Number of unique product strings: {len(uniqueProductStrings_high)}")
print(f"Number of unique individual reactant molecules: {len(uniqueReactantMolecules_high)}")
print(f"Number of unique individual product molecules: {len(uniqueProductMolecules_high)}")

reactionDF_DORAXGB_highFeasibility.to_csv(os.path.join(DNADesignResultsDir, f"DORAXGB_highFeasibility_{feasibilityLabel}.csv"),index=False)
reactionDF_DORAXGB_highFeasibility.head()

Feasibility label used: feasibilityLabel_rule2 == 1
Number of high-feasibility reactions: 4733
Number of unique reactant strings: 285
Number of unique product strings: 227
Number of unique individual reactant molecules: 167
Number of unique individual product molecules: 192


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,connectionMoleculesByStep,rxn_str,feasibilityScore_rule1,feasibilityLabel_rule1,feasibilityScore_rule2,feasibilityLabel_rule2,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4
0,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,No_Thermo,"(1, 1, 1, 1)","(1, 1, 1)",Enzymatic,4,3,...,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,0.166996,0.0,0.967761,1.0,0.922012,1.0,0.886861,1.0
1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,0.002562,0.0,0.738403,1.0,0.627660,1.0,0.740460,1.0
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,0.297114,0.0,0.995089,1.0,0.924887,1.0,0.990581,1.0
3,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,0.297114,0.0,0.995089,1.0,0.924887,1.0,0.990581,1.0
4,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,0.297114,0.0,0.995089,1.0,0.924887,1.0,0.990581,1.0


### Summarize unique DORAnet rules

Many reactions may use the same rule. We do not want to annotate millions of reactions one by one. First annotate the rules.

In [29]:
enzymaticReactionDF = reactionDF_DORAXGB_highFeasibility.copy()
ruleSummaryDF = (
    enzymaticReactionDF
    .groupby(["ruleName", "reactionType"], dropna=False)
    .agg(
        numReactions=("reactionString", "count"),
        #numStarterSources=("sourceFolderNum", "nunique"),
        exampleReaction=("reactionString", "first"),
        exampleReactants=("reactants", "first"),
        exampleProducts=("products", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)

print(f"unique DORAnet rules    : {enzymaticReactionDF['ruleName'].nunique():,}")
ruleSummaryDF

unique DORAnet rules    : 49


,ruleName,reactionType,numReactions,exampleReaction,exampleReactants,exampleProducts
44,rule0165_2,Enzymatic,649,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...
20,rule0017_16,Enzymatic,592,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1OC(CO)[C@@...,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1OC(CO)[C@@...,O.O=c1[nH]cnc2c1ncn2[C@@H]1OC(COP(=O)(O)O)[C@@...
31,rule0062_17,Enzymatic,553,CC(=O)O.CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)...,CC(=O)O.CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)...,CC(=O)OC(C)=O.CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]...
35,rule0073_5,Enzymatic,511,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...
8,rule0007_198,Enzymatic,452,CC(=O)O[C@@H]1[C@H](O)[C@@H](CO)O[C@]1(C)n1cnc...,CC(=O)O[C@@H]1[C@H](O)[C@@H](CO)O[C@]1(C)n1cnc...,CC(=O)O.C[C@]1(n2cnc3c(=O)[nH]cnc32)O[C@H](CO)...
30,rule0061_4,Enzymatic,330,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...
4,rule0007_161,Enzymatic,239,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,CC(=O)O.CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]...
48,rule0491_2,Enzymatic,238,CC(O)O[C@@H]1[C@H]2CO[C@@]1(O)[C@H](n1cnc3c(=O...,CC(O)O[C@@H]1[C@H]2CO[C@@]1(O)[C@H](n1cnc3c(=O...,CC(=O)O.NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(...
32,rule0062_18,Enzymatic,154,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...
33,rule0062_19,Enzymatic,153,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...


### Add required placeholder columns

In [30]:
enzymeAnnotationDF = ruleSummaryDF.copy()

enzymeAnnotationDF["suggestedEnzymeClass"] = ""
enzymeAnnotationDF["ecNumber"] = ""
enzymeAnnotationDF["enzymeName"] = ""
enzymeAnnotationDF["uniprotAccession"] = ""
enzymeAnnotationDF["sourceDatabase"] = ""
enzymeAnnotationDF["reactionSimilarity"] = ""
enzymeAnnotationDF["enzymeConfidence"] = ""
enzymeAnnotationDF["notes"] = ""
enzymeAnnotationDF.head()

,ruleName,reactionType,numReactions,exampleReaction,exampleReactants,exampleProducts,suggestedEnzymeClass,ecNumber,enzymeName,uniprotAccession,sourceDatabase,reactionSimilarity,enzymeConfidence,notes
44,rule0165_2,Enzymatic,649,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,,,,,,,,
20,rule0017_16,Enzymatic,592,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1OC(CO)[C@@...,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1OC(CO)[C@@...,O.O=c1[nH]cnc2c1ncn2[C@@H]1OC(COP(=O)(O)O)[C@@...,,,,,,,,
31,rule0062_17,Enzymatic,553,CC(=O)O.CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)...,CC(=O)O.CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)...,CC(=O)OC(C)=O.CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]...,,,,,,,,
35,rule0073_5,Enzymatic,511,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,,,,,,,,
8,rule0007_198,Enzymatic,452,CC(=O)O[C@@H]1[C@H](O)[C@@H](CO)O[C@]1(C)n1cnc...,CC(=O)O[C@@H]1[C@H](O)[C@@H](CO)O[C@]1(C)n1cnc...,CC(=O)O.C[C@]1(n2cnc3c(=O)[nH]cnc32)O[C@H](CO)...,,,,,,,,


## 3. Read DORAnet reaction ruleset file to extract `UniProt ID` for each `reaction rule`

In [31]:
doranetRulesetDF = pd.read_csv(dataDir + "/DORAnet/JN3604IMT_rules.tsv", sep="\t")

print(doranetRulesetDF.shape)
print(doranetRulesetDF.columns.tolist())
doranetRulesetDF.head()

(3604, 5)
['Name', 'Reactants', 'SMARTS', 'Products', 'Comments']


,Name,Reactants,SMARTS,Products,Comments
0,rule0001_01,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,P97259;P97402;Q08834;Q09328;Q16408;Q3V5L5;Q765...
1,rule0001_02,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,A0K8D0;A0KHH6;A0KZ10;A0L8R9;A0Q7X9;A1A7M6;A1K6...
2,rule0001_03,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,P0DMP6;P0DMP7;P16442;Q15951;Q1RLK6;Q3USF0;Q5HZ...
3,rule0001_04,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,A4IID1;O19071;O77836;P23336;P26572;P27115;P278...
4,rule0001_05,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,NaN


### Find the number of common reaction rules between `DORAnet` diversification and ruleset table from actual package

In [32]:
reactionRuleSet = set(reactionDF_DORAXGB_highFeasibility["ruleName"].dropna().astype(str).str.strip())
tsvRuleSet = set(doranetRulesetDF["Name"].dropna().astype(str).str.strip())

matchedRuleSet = reactionRuleSet.intersection(tsvRuleSet)
missingRuleSet = reactionRuleSet.difference(tsvRuleSet)

print(f"Rules present in DORAnet diversification run: {len(reactionRuleSet):,}")
print(f"Rules matched from DORAnet package: {len(matchedRuleSet):,}")
print(f"Rules missing from DORAnet package: {len(missingRuleSet):,}")

print("\nMatched rules:")
print(sorted(list(matchedRuleSet))[:20])

print("\nMissing rules:")
print(sorted(list(missingRuleSet))[:20])

Rules present in DORAnet diversification run: 49
Rules matched from DORAnet package: 49
Rules missing from DORAnet package: 0

Matched rules:
['rule0002_154', 'rule0003_171', 'rule0003_176', 'rule0003_177', 'rule0007_161', 'rule0007_174', 'rule0007_193', 'rule0007_195', 'rule0007_198', 'rule0007_199', 'rule0007_200', 'rule0007_202', 'rule0007_203', 'rule0010_65', 'rule0011_50', 'rule0011_51', 'rule0015_21', 'rule0016_061', 'rule0016_062', 'rule0016_063']

Missing rules:
[]


### Extract `UniProt IDs` from `DORAnet` reaction rules

In [33]:
def splitUniProtIds(idString):
    if pd.isna(idString):
        return []
    return [x.strip() for x in str(idString).split(";") if x.strip()]


ruleInfoDF = doranetRulesetDF.copy()

ruleInfoDF = ruleInfoDF.rename(columns={
    "Name": "ruleName",
    "Reactants": "ruleReactants",
    "SMARTS": "ruleSMARTS",
    "Products": "ruleProducts",
    "Comments": "candidateUniProtRaw",
})

ruleInfoDF["ruleName"] = ruleInfoDF["ruleName"].astype(str).str.strip()
ruleInfoDF["candidateUniProtList"] = ruleInfoDF["candidateUniProtRaw"].apply(splitUniProtIds)
ruleInfoDF["numCandidateUniProt"] = ruleInfoDF["candidateUniProtList"].apply(len)

ruleInfoDF = ruleInfoDF[
    [
        "ruleName",
        "ruleReactants",
        "ruleProducts",
        "ruleSMARTS",
        "candidateUniProtRaw",
        "numCandidateUniProt",
    ]
].copy()



print(f"ruleInfoDF rows: {len(ruleInfoDF):,}")
ruleInfoDF

ruleInfoDF rows: 3,604


,ruleName,ruleReactants,ruleProducts,ruleSMARTS,candidateUniProtRaw,numCandidateUniProt
0,rule0001_01,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,P97259;P97402;Q08834;Q09328;Q16408;Q3V5L5;Q765...,10
1,rule0001_02,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,A0K8D0;A0KHH6;A0KZ10;A0L8R9;A0Q7X9;A1A7M6;A1K6...,263
2,rule0001_03,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,P0DMP6;P0DMP7;P16442;Q15951;Q1RLK6;Q3USF0;Q5HZ...,29
3,rule0001_04,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,A4IID1;O19071;O77836;P23336;P26572;P27115;P278...,42
4,rule0001_05,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,NaN,0
...,...,...,...,...,...,...
3599,rule1152_1,Any;Any,Any;Any,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...,Q8ZUN8,1
3600,rule1152_2,Any;Any,Any;Any,[#6;!$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[...,B9TTF1,1
3601,rule1164_1,Any;WATER,Any;H2O2,[#6;$([#6&!R]-&!@[#6&R]1:&@[#6&R]:&@[#6&R]:&@[...,Q972I2,1
3602,rule1165_1,Any;H2O2,Any;WATER,[#6;$([#6&!R]-[#6&R]1:&@[#6&R]:&@[#6&R]:&@[#6&...,Q972I2,1


## 4. Merge rule information into `reactionDF`

This connects DORAnet reactions with the rule SMARTS and UniProt candidate list.

In [34]:
reactionDF_wUniprotID = reactionDF_DORAXGB_highFeasibility.merge(ruleInfoDF,on="ruleName",how="left")

reactionDF_wUniprotID["hasRuleLookup"] = reactionDF_wUniprotID["ruleSMARTS"].notna()

print(reactionDF_wUniprotID["hasRuleLookup"].value_counts(dropna=False))
reactionDF_wUniprotID

hasRuleLookup
True    4733
Name: count, dtype: int64


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4,ruleReactants,ruleProducts,ruleSMARTS,candidateUniProtRaw,numCandidateUniProt,hasRuleLookup
0,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,No_Thermo,"(1, 1, 1, 1)","(1, 1, 1)",Enzymatic,4,3,...,0.922012,1.0,0.886861,1.0,Any;NAD_CoF;NAD_CoF;WATER,NADH_CoF;NADH_CoF;Any,[#6;$([#6&!R]-[#8&!R]);!$([#6&!R](-[#6&R]1-&@[...,A0A0J9X7D2;A1BPP9;A4IP64;A4ISB9;B0S9F2;B0SS41;...,228,True
1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,0.627660,1.0,0.740460,1.0,Any;NADH_CoF;O2,NAD_CoF;Any;WATER;WATER,[#6;$([#6&!R]-&!@[#6&R]);!$([#6&!R]-&!@[#6&R]1...,E3VWJ1;F1T282;F1T283;I1TEM3;O94142;Q16850;Q2MJ...,11,True
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.924887,1.0,0.990581,1.0,CoA;Any,ACETYL-COA;Any,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True
3,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.924887,1.0,0.990581,1.0,CoA;Any,ACETYL-COA;Any,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True
4,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.924887,1.0,0.990581,1.0,CoA;Any,ACETYL-COA;Any,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4728,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,0.627660,1.0,0.740460,1.0,Any;NADH_CoF;O2,NAD_CoF;Any;WATER;WATER,[#6;$([#6&!R]-&!@[#6&R]);!$([#6&!R]-&!@[#6&R]1...,E3VWJ1;F1T282;F1T283;I1TEM3;O94142;Q16850;Q2MJ...,11,True
4729,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.924887,1.0,0.990581,1.0,CoA;Any,ACETYL-COA;Any,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True
4730,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.924887,1.0,0.990581,1.0,CoA;Any,ACETYL-COA;Any,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True
4731,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.924887,1.0,0.990581,1.0,CoA;Any,ACETYL-COA;Any,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True


## 4. Find best `UniProt ID` per rule by `Reaction Center Morgan Fingerprint (RCMFP)` enzyme retrieval similarity search 

Implements workflow from the DORAnet/DORA-XGB → atom mapping → RCMFP → known enzyme retrieval. It uses `ruleBase` as the practical bridge between expanded DORAnet rule names such as `rule0003_170` and known reference coarse operator IDs such as `rule0003`, then ranks candidate natural enzyme precedents by RCMFP Tanimoto similarity.

Use this tool: https://github.com/stefanpate/ergochemics/tree/main#

### Patch `RDKit/ergochemics` compatibility

This preserves atom-map labels. Do not use a patch that clears atom-map numbers, because RCMFP needs atom maps to infer reaction centers.

In [35]:
# User paths
ERGOCHEMICS_SRC = "/users/sghosh6/DTRA_project/MACAW/ergochemics/src"
if ERGOCHEMICS_SRC not in sys.path: sys.path.insert(0, ERGOCHEMICS_SRC)

import ergochemics.mapping as egmap
import ergochemics.similarity as egsim
print("RDKit version:", rdBase.rdkitVersion); print("ergochemics mapping file:", egmap.__file__)

_RDKit_MolToSmiles_original = rdmolfiles.MolToSmiles

def MolToSmiles_compat(mol, *args, **kwargs):
    kwargs.pop("ignoreAtomMapNumbers", None)
    return _RDKit_MolToSmiles_original(mol, *args, **kwargs)

Chem.MolToSmiles = MolToSmiles_compat; egmap.Chem.MolToSmiles = MolToSmiles_compat; egsim.Chem.MolToSmiles = MolToSmiles_compat
operator_map_reaction, get_reaction_center = egmap.operator_map_reaction, egmap.get_reaction_center
ReactionFingerprinter, MolFeaturizer = egsim.ReactionFingerprinter, egsim.MolFeaturizer
print("Patch test:", Chem.MolToSmiles(Chem.MolFromSmiles("[CH3:1][OH:2]"), ignoreAtomMapNumbers=True))

RDKit version: 2023.09.5
ergochemics mapping file: /users/sghosh6/DTRA_project/MACAW/ergochemics/src/ergochemics/mapping.py
Patch test: [CH3:1][OH:2]


### Normalize DORAnet reaction strings and `atom-map` all reactions

In [36]:
def normalize_reaction_string(rxn):
    if pd.isna(rxn): return None
    rxn = str(rxn).strip().replace(" ", "")
    if rxn.count(">") == 2 and ">>" in rxn: return rxn
    parts = rxn.split(">")
    return f"{parts[0]}>>{parts[2]}" if len(parts) == 3 else None

def map_query_reaction_with_rule(row):
    rxn, ruleSMARTS = row.get("queryReaction"), row.get("ruleSMARTS")
    if pd.isna(rxn) or str(rxn).strip() == "": return None, "missing_queryReaction"
    if pd.isna(ruleSMARTS) or str(ruleSMARTS).strip() == "": return None, "missing_ruleSMARTS"
    rxn, ruleSMARTS, lastError = str(rxn).replace(" ", ""), str(ruleSMARTS).strip(), None
    if ">>" not in rxn: return None, "invalid_queryReaction_no_double_arrow"
    if ">>" not in ruleSMARTS: return None, "invalid_ruleSMARTS_no_double_arrow"
    for explicitHsFlag, statusLabel in [(False, "mapped"), (True, "mapped_explicit_h")]:
        try:
            result = operator_map_reaction(rxn=rxn, operator=ruleSMARTS, explicit_hs=explicitHsFlag, quiet=True)
            if result.did_map and result.atom_mapped_smarts is not None: return result.atom_mapped_smarts, statusLabel
            lastError = f"{statusLabel}_failed"
        except Exception as exc: lastError = f"mapping_error:{type(exc).__name__}:{exc}"
    return None, lastError

AtomMapDF = reactionDF_wUniprotID.copy()
AtomMapDF["queryReaction"] = AtomMapDF["reactionString"].apply(normalize_reaction_string)
AtomMapDF = AtomMapDF[AtomMapDF["queryReaction"].notna() & AtomMapDF["ruleSMARTS"].notna()].drop(columns=["queryMappedReaction", "rcmfpMappingStatus"], errors="ignore").copy()

mappedPairs = AtomMapDF.progress_apply(map_query_reaction_with_rule, axis=1)
AtomMapDF[["queryMappedReaction", "rcmfpMappingStatus"]] = pd.DataFrame(mappedPairs.tolist(), index=AtomMapDF.index)
sucessAtomMapDF = AtomMapDF[AtomMapDF["queryMappedReaction"].notna()].copy() 
failedAtomMapDF = AtomMapDF[AtomMapDF["queryMappedReaction"].isna()].copy()
print(AtomMapDF["rcmfpMappingStatus"].value_counts(dropna=False))
print("Successful atom mapping:", len(sucessAtomMapDF), "| Failed:", len(failedAtomMapDF))
sucessAtomMapDF.shape

100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 4733/4733 [00:23<00:00, 203.48it/s]

rcmfpMappingStatus
mapped    4733
Name: count, dtype: int64
Successful atom mapping: 4733 | Failed: 0


(4733, 54)

### Compute `RCMFP fingerprints`

In [37]:
TOP_K, MIN_SIMILARITY, FP_SIDE_LENGTH, FP_RADIUS = 20, 0.0, 2048, 2

In [38]:
reactionFingerprinter = ReactionFingerprinter(radius=FP_RADIUS, length=FP_SIDE_LENGTH, mol_featurizer=MolFeaturizer())

def compute_rcmfp_with_status(mappedRxn):
    try:
        if pd.isna(mappedRxn) or str(mappedRxn).strip() == "": return None, "missing_mapped_reaction"
        mappedRxn = str(mappedRxn).replace(" ", "")
        if ">>" not in mappedRxn: return None, "invalid_mapped_reaction_no_double_arrow"
        if not pd.Series([mappedRxn]).str.contains(r":\d+\]", regex=True).iloc[0]: return None, "no_atom_map_labels"
        lrc, rrc = get_reaction_center(mappedRxn, mode="combined")
        if len(lrc) == 0 or len(rrc) == 0: return None, f"empty_reaction_center:lrc={len(lrc)}:rrc={len(rrc)}"
        fp = reactionFingerprinter.fingerprint(mappedRxn, output_type="bit", use_rc=True, rc_dist_ub=None)
        return fp.astype(bool), "rcmfp_success"
    except Exception as exc: return None, f"rcmfp_error:{type(exc).__name__}:{exc}"


rcmfpPairs = sucessAtomMapDF["queryMappedReaction"].progress_apply(compute_rcmfp_with_status)
sucessAtomMapDF[["queryRCMFP", "rcmfpStatus"]] = pd.DataFrame(rcmfpPairs.tolist(), index=sucessAtomMapDF.index)
queryRcmfpFailureDF = sucessAtomMapDF[sucessAtomMapDF["queryRCMFP"].isna()].copy()
sucessAtomMapDF = sucessAtomMapDF[sucessAtomMapDF["queryRCMFP"].notna()].copy().reset_index(drop=True)
sucessAtomMapDF

  0%|                                                                                                             | 0/4733 [00:00<?, ?it/s]

{0: 1, 1: 2, 2: 3, 3: 10, 4: 11, 5: 13, 6: 16, 7: 18, 8: 19, 9: 20, 10: 21, 11: 22, 12: 23, 13: 24, 14: 25, 15: 26, 16: 27, 17: 28, 18: 29, 19: 31, 20: 34, 21: 36, 22: 39, 23: 41, 24: 37, 25: 40, 26: 45, 27: 47, 28: 46, 29: 48, 30: 32, 31: 35, 32: 38, 33: 42, 34: 43, 35: 44, 36: 30, 37: 33, 38: 14, 39: 17, 40: 12, 41: 15, 42: 4, 43: 5, 44: 7, 45: 8, 46: 9, 47: 6, 48: 49, 49: 55, 50: 56, 51: 57, 52: 50, 53: 51, 54: 58, 55: 59, 56: 61, 57: 64, 58: 66, 59: 67, 60: 68, 61: 69, 62: 70, 63: 71, 64: 72, 65: 73, 66: 74, 67: 75, 68: 76, 69: 77, 70: 79, 71: 82, 72: 84, 73: 87, 74: 89, 75: 85, 76: 88, 77: 93, 78: 95, 79: 94, 80: 96, 81: 80, 82: 83, 83: 86, 84: 90, 85: 91, 86: 92, 87: 78, 88: 81, 89: 62, 90: 65, 91: 60, 92: 63, 93: 52, 94: 53, 95: 54, 96: 97, 97: 98, 98: 99, 99: 101, 100: 103, 101: 106, 102: 108, 103: 110, 104: 112, 105: 109, 106: 111, 107: 113, 108: 115, 109: 114, 110: 116, 111: 105, 112: 107, 113: 102, 114: 104, 115: 100}
[0, 1, 2, 42, 43, 47, 44, 45, 46, 3, 4, 40, 5, 38, 41, 6,

  0%|                                                                                                     | 3/4733 [00:00<11:10,  7.06it/s]

{0: 1, 1: 7, 2: 8, 3: 9, 4: 2, 5: 3, 6: 4, 7: 5, 8: 10, 9: 11, 10: 13, 11: 16, 12: 18, 13: 19, 14: 20, 15: 21, 16: 22, 17: 23, 18: 24, 19: 25, 20: 26, 21: 27, 22: 28, 23: 29, 24: 31, 25: 34, 26: 36, 27: 39, 28: 41, 29: 37, 30: 40, 31: 45, 32: 47, 33: 46, 34: 48, 35: 32, 36: 35, 37: 38, 38: 42, 39: 43, 40: 44, 41: 30, 42: 33, 43: 14, 44: 17, 45: 12, 46: 15, 47: 6, 48: 49, 49: 50, 50: 51, 51: 52, 52: 54, 53: 57, 54: 59, 55: 61, 56: 63, 57: 60, 58: 62, 59: 64, 60: 66, 61: 65, 62: 67, 63: 56, 64: 58, 65: 53, 66: 55, 67: 68, 68: 69}
[0, 4, 5, 6, 7, 47, 1, 2, 3, 8, 9, 45, 10, 43, 46, 11, 44, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 41, 24, 35, 42, 25, 36, 26, 29, 37, 27, 30, 28, 38, 39, 40, 31, 33, 32, 34, 48, 49, 50, 51, 65, 52, 66, 63, 53, 64, 54, 57, 55, 58, 56, 59, 61, 60, 62, 67, 68]
[1, 2, 3, 4, 5, 6, 49, 50, 68, 69]
{0: 1, 1: 7, 2: 8, 3: 9, 4: 2, 5: 3, 6: 4, 7: 5, 8: 10, 9: 11, 10: 13, 11: 16, 12: 18, 13: 19, 14: 20, 15: 21, 16: 22, 17: 23, 18: 24, 19: 25, 20: 26, 21: 27, 22: 2

  0%|▏                                                                                                    | 9/4733 [00:00<04:34, 17.19it/s]

...

{0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 9, 8: 10, 9: 11, 10: 12, 11: 14, 12: 16, 13: 17, 14: 18, 15: 19, 16: 20, 17: 21, 18: 22, 19: 23, 20: 24, 21: 25, 22: 26, 23: 27, 24: 28, 25: 29, 26: 30, 27: 32, 28: 35, 29: 38, 30: 43, 31: 45, 32: 39, 33: 44, 34: 46, 35: 48, 36: 47, 37: 49, 38: 33, 39: 36, 40: 31, 41: 34, 42: 37, 43: 40, 44: 41, 45: 42, 46: 15, 47: 13, 48: 8, 49: 50, 50: 51, 51: 52, 52: 53, 53: 54, 54: 55, 55: 57, 56: 60, 57: 62, 58: 64, 59: 66, 60: 63, 61: 65, 62: 67, 63: 69, 64: 68, 65: 70, 66: 58, 67: 61, 68: 56, 69: 59}
[0, 1, 2, 3, 4, 5, 6, 48, 7, 8, 9, 10, 47, 11, 46, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 40, 27, 38, 41, 28, 39, 42, 29, 32, 43, 44, 45, 30, 33, 31, 34, 36, 35, 37, 49, 50, 51, 52, 53, 54, 68, 55, 66, 69, 56, 67, 57, 60, 58, 61, 59, 62, 64, 63, 65]
[1, 2, 52]
{0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 9, 8: 10, 9: 11, 10: 12, 11: 14, 12: 16, 13: 17, 14: 18, 15: 19, 16: 20, 17: 21, 18: 22, 19: 23, 20: 24, 21: 25, 22: 26, 23: 27, 2

100%|█████████████████████████████████████████████████████████████████████████████████████████████████▉| 4731/4733 [06:56<00:00, 17.97it/s]

{0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 9, 8: 10, 9: 11, 10: 12, 11: 14, 12: 16, 13: 17, 14: 18, 15: 19, 16: 20, 17: 21, 18: 22, 19: 23, 20: 24, 21: 25, 22: 26, 23: 27, 24: 28, 25: 29, 26: 30, 27: 32, 28: 35, 29: 38, 30: 43, 31: 45, 32: 39, 33: 44, 34: 46, 35: 48, 36: 47, 37: 49, 38: 33, 39: 36, 40: 31, 41: 34, 42: 37, 43: 40, 44: 41, 45: 42, 46: 15, 47: 13, 48: 8, 49: 50, 50: 51, 51: 52, 52: 53, 53: 54, 54: 55, 55: 57, 56: 60, 57: 62, 58: 64, 59: 66, 60: 63, 61: 65, 62: 67, 63: 69, 64: 68, 65: 70, 66: 58, 67: 61, 68: 56, 69: 59}
[0, 1, 2, 3, 4, 5, 6, 48, 7, 8, 9, 10, 47, 11, 46, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 40, 27, 38, 41, 28, 39, 42, 29, 32, 43, 44, 45, 30, 33, 31, 34, 36, 35, 37, 49, 50, 51, 52, 53, 54, 68, 55, 66, 69, 56, 67, 57, 60, 58, 61, 59, 62, 64, 63, 65]
[1, 2, 52]
{0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 9, 8: 10, 9: 11, 10: 12, 11: 14, 12: 16, 13: 17, 14: 18, 15: 19, 16: 20, 17: 21, 18: 22, 19: 23, 20: 24, 21: 25, 22: 26, 23: 27, 2

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 4733/4733 [06:56<00:00, 11.36it/s]


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,ruleProducts,ruleSMARTS,candidateUniProtRaw,numCandidateUniProt,hasRuleLookup,queryReaction,queryMappedReaction,rcmfpMappingStatus,queryRCMFP,rcmfpStatus
0,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,No_Thermo,"(1, 1, 1, 1)","(1, 1, 1)",Enzymatic,4,3,...,NADH_CoF;NADH_CoF;Any,[#6;$([#6&!R]-[#8&!R]);!$([#6&!R](-[#6&R]1-&@[...,A0A0J9X7D2;A1BPP9;A4IP64;A4ISB9;B0S9F2;B0SS41;...,228,True,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,[CH2:97]([CH:99]1[O:101][CH:103]([n:106]2[cH:1...,mapped,"[False, False, False, False, False, False, Fal...",rcmfp_success
1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,NAD_CoF;Any;WATER;WATER,[#6;$([#6&!R]-&!@[#6&R]);!$([#6&!R]-&!@[#6&R]1...,E3VWJ1;F1T282;F1T283;I1TEM3;O94142;Q16850;Q2MJ...,11,True,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,[CH2:49]([OH:50])[CH:51]1[O:52][CH:54]([n:57]2...,mapped,"[False, False, False, False, False, False, Fal...",rcmfp_success
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,ACETYL-COA;Any,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,[SH:2][CH2:3][CH2:4][NH:5][C:6]([CH2:7][CH2:9]...,mapped,"[False, False, True, False, False, False, Fals...",rcmfp_success
3,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,ACETYL-COA;Any,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,[SH:2][CH2:3][CH2:4][NH:5][C:6]([CH2:7][CH2:9]...,mapped,"[False, False, True, False, False, False, Fals...",rcmfp_success
4,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,ACETYL-COA;Any,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,[SH:2][CH2:3][CH2:4][NH:5][C:6]([CH2:7][CH2:9]...,mapped,"[False, False, True, False, False, False, Fals...",rcmfp_success
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4728,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,NAD_CoF;Any;WATER;WATER,[#6;$([#6&!R]-&!@[#6&R]);!$([#6&!R]-&!@[#6&R]1...,E3VWJ1;F1T282;F1T283;I1TEM3;O94142;Q16850;Q2MJ...,11,True,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,[CH2:49]([OH:50])[CH:51]1[O:52][CH:54]([n:57]2...,mapped,"[False, False, False, False, False, False, Fal...",rcmfp_success
4729,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,ACETYL-COA;Any,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,[SH:2][CH2:3][CH2:4][NH:5][C:6]([CH2:7][CH2:9]...,mapped,"[False, False, True, False, False, False, Fals...",rcmfp_success
4730,CC(=O)OC[C@H]1O[C@@H]

In [39]:
print("RCMFP status counts:")
print(sucessAtomMapDF["rcmfpStatus"].value_counts(dropna=False))
print("Valid RCMFP reactions:", len(sucessAtomMapDF))
print("Failed RCMFP reactions:", len(queryRcmfpFailureDF))

RCMFP status counts:
rcmfpStatus
rcmfp_success    4733
Name: count, dtype: int64
Valid RCMFP reactions: 4733
Failed RCMFP reactions: 0


### Load and align known enzyme reference metadata and RCMFP cache

- loaded from: https://github.com/JBEI/TridentSynthWeb/tree/b17e61040b182ce68b2931f8dbbaf65f370f8c03/data/processed

In [40]:
naturalEnzymeFilePath = os.path.join(dataDir + "/DORAnet/known_enzyme_reactions_union.parquet")
naturalEnzyme_FP_FilePath = os.path.join(dataDir + "/DORAnet/known_rxn_rcmfp_cache.npz")

naturalEnzymeDF = pd.read_parquet(naturalEnzymeFilePath)
naturalEnzymeDF_wFingerprints = np.load(naturalEnzyme_FP_FilePath, allow_pickle=True)
naturalEnzyme_fingerprint_matrix = naturalEnzymeDF_wFingerprints["fingerprints"].astype(bool)
naturalEnzyme_RC_PatternLHS, naturalEnzyme_RC_PatternRHS, knownValidIndices = naturalEnzymeDF_wFingerprints["rc_patterns_lhs"], naturalEnzymeDF_wFingerprints["rc_patterns_rhs"], naturalEnzymeDF_wFingerprints["valid_indices"]
naturalEnzymeReferenceDF = naturalEnzymeDF.iloc[knownValidIndices].reset_index(drop=True).copy()
naturalEnzymeReferenceDF["naturalEnzyme_RC_PatternLHS"], naturalEnzymeReferenceDF["naturalEnzyme_RC_PatternRHS"], naturalEnzymeReferenceDF["knownOriginalIndex"] = naturalEnzyme_RC_PatternLHS, naturalEnzyme_RC_PatternRHS, knownValidIndices
assert len(naturalEnzymeReferenceDF) == naturalEnzyme_fingerprint_matrix.shape[0]
print("Known enzyme reference shape:", naturalEnzymeReferenceDF.shape); print("Known enzyme molecular fingerprint matrix shape:", naturalEnzyme_fingerprint_matrix.shape)

Known enzyme reference shape: (54063, 25)
Known enzyme molecular fingerprint matrix shape: (54063, 4096)


### Compute RCMFP fingerprint matrices for `DORAnet` reactions

In [41]:
def make_reverse_fingerprint_matrix(fpMatrix, sideLength=FP_SIDE_LENGTH): return np.hstack([fpMatrix[:, sideLength:], fpMatrix[:, :sideLength]]).astype(bool)
def make_fingerprint_matrix_from_column(df, fpCol):
    validMask = df[fpCol].notna(); metaDF = df.loc[validMask].drop(columns=[fpCol], errors="ignore").reset_index(drop=True)
    fpMatrix = np.vstack(df.loc[validMask, fpCol].to_numpy()).astype(bool); fpReverseMatrix = make_reverse_fingerprint_matrix(fpMatrix)
    return metaDF, fpMatrix, fpReverseMatrix, fpMatrix.sum(axis=1), fpReverseMatrix.sum(axis=1)

queryMetaDF, query_FP_Matrix, query_FP_MatrixReverse, queryPopcounts, queryReversePopcounts = make_fingerprint_matrix_from_column(sucessAtomMapDF, "queryRCMFP")
naturalEnzyme_fingerprint_matrixReverse = make_reverse_fingerprint_matrix(naturalEnzyme_fingerprint_matrix)
knownPopcounts, knownReversePopcounts = naturalEnzyme_fingerprint_matrix.sum(axis=1), naturalEnzyme_fingerprint_matrixReverse.sum(axis=1)

queryKeepCols = ["reactants", "products", "reactionString", "ruleName", "reactionType", "rxn_str", "ruleSMARTS", "candidateUniProtRaw", "numCandidateUniProt", "hasRuleLookup", "queryReaction", "queryMappedReaction", "rcmfpMappingStatus", "rcmfpStatus", "feasibilityScore_rule1", "feasibilityLabel_rule1", "feasibilityScore_rule2", "feasibilityLabel_rule2", "feasibilityScore_rule3", "feasibilityLabel_rule3", "feasibilityScore_rule4", "feasibilityLabel_rule4"]
queryMetaDF = queryMetaDF[[c for c in queryKeepCols if c in queryMetaDF.columns]].copy()
print("Query metadata shape:", queryMetaDF.shape); print("Query molecular fingerprint matrix shape:", query_FP_Matrix.shape); print("Known enzyme molecular fingerprint matrix shape:", naturalEnzyme_fingerprint_matrix.shape)
queryMetaDF

Query metadata shape: (4733, 22)
Query molecular fingerprint matrix shape: (4733, 4096)
Known enzyme molecular fingerprint matrix shape: (54063, 4096)


,reactants,products,reactionString,ruleName,reactionType,rxn_str,ruleSMARTS,candidateUniProtRaw,numCandidateUniProt,hasRuleLookup,...,rcmfpMappingStatus,rcmfpStatus,feasibilityScore_rule1,feasibilityLabel_rule1,feasibilityScore_rule2,feasibilityLabel_rule2,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4
0,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,Enzymatic,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,[#6;$([#6&!R]-[#8&!R]);!$([#6&!R](-[#6&R]1-&@[...,A0A0J9X7D2;A1BPP9;A4IP64;A4ISB9;B0S9F2;B0SS41;...,228,True,...,mapped,rcmfp_success,0.166996,0.0,0.967761,1.0,0.922012,1.0,0.886861,1.0
1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,Enzymatic,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,[#6;$([#6&!R]-&!@[#6&R]);!$([#6&!R]-&!@[#6&R]1...,E3VWJ1;F1T282;F1T283;I1TEM3;O94142;Q16850;Q2MJ...,11,True,...,mapped,rcmfp_success,0.002562,0.0,0.738403,1.0,0.627660,1.0,0.740460,1.0
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,Enzymatic,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True,...,mapped,rcmfp_success,0.297114,0.0,0.995089,1.0,0.924887,1.0,0.990581,1.0
3,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,Enzymatic,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True,...,mapped,rcmfp_success,0.297114,0.0,0.995089,1.0,0.924887,1.0,0.990581,1.0
4,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,Enzymatic,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True,...,mapped,rcmfp_success,0.297114,0.0,0.995089,1.0,0.924887,1.0,0.990581,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4728,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,Enzymatic,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,[#6;$([#6&!R]-&!@[#6&R]);!$([#6&!R]-&!@[#6&R]1...,E3VWJ1;F1T282;F1T283;I1TEM3;O94142;Q16850;Q2MJ...,11,True,...,mapped,rcmfp_success,0.002562,0.0,0.738403,1.0,0.627660,1.0,0.740460,1.0
4729,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,Enzymatic,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True,...,mapped,rcmfp_success,0.297114,0.0,0.995089,1.0,0.924887,1.0,0.990581,1.0
4730,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,Enzymatic,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,True,...,mapped,rcmfp_success,0.297114,0.0,0.995089,1.0,0.924887,1.0,0.990581,1.0
4731,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,Enzymatic,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,[#16:1].[#6;$([#6&!R](-&!@[#6&!R])=&!@[#8&!R])...,A0A0B4VSM7;A0

### Add `ruleBase` column to known enzyme reference and query metadata from DORAnet reactions

This avoids forcing known reference rules into expanded DORAnet rule names. It uses coarse operator family IDs such as `rule0003` and then lets RCMFP decide the closest natural reaction within that family.

In [42]:
def get_rule_base(x): return None if pd.isna(x) else str(x).strip().split("_")[0]
def parse_operator_list(x):
    if isinstance(x, np.ndarray): return [str(v).strip() for v in x.tolist() if v is not None and str(v).strip()]
    if isinstance(x, (list, tuple, set)): return [str(v).strip() for v in list(x) if v is not None and str(v).strip()]
    if x is None or pd.isna(x): return []
    s = str(x).strip()
    if s in ["", "nan", "None", "[]"]: return []
    try:
        parsed = ast.literal_eval(s)
        return [str(v).strip() for v in list(parsed)] if isinstance(parsed, (list, tuple, set, np.ndarray)) else [str(parsed).strip()]
    except Exception:
        for sep in ["|", ";", ","]:
            if sep in s: return [v.strip().strip("'\"") for v in s.split(sep) if v.strip()]
        return [s]

def choose_known_rule_base(row):
    top = row.get("top_mapped_operator")
    if top is not None and not pd.isna(top) and str(top).strip().startswith("rule"): return str(top).strip(), "top_mapped_operator"
    for op in parse_operator_list(row.get("all_mapped_operators")):
        if str(op).startswith("rule"): return str(op), "all_mapped_operators"
    return None, "no_ruleBase"



# Build ruleBase in naturalEnzymeReferenceDF
naturalEnzymeReferenceDF = naturalEnzymeReferenceDF.copy()

naturalEnzymeReferenceDF[["ruleBase", "ruleBaseSource"]] = naturalEnzymeReferenceDF.apply(
    lambda row: pd.Series(choose_known_rule_base(row)),
    axis=1
)

# If you want only base (e.g., rule0001 from rule0001_22), normalize:
naturalEnzymeReferenceDF["ruleBase"] = naturalEnzymeReferenceDF["ruleBase"].apply(get_rule_base)

# Rebuild metadata
queryMetaDF = queryMetaDF.reset_index(drop=True).copy()
knownMetaDF = naturalEnzymeReferenceDF.reset_index(drop=True).copy()

queryMetaDF["queryRowId"] = np.arange(len(queryMetaDF))
knownMetaDF["knownRowId"] = np.arange(len(knownMetaDF))

queryMetaDF["ruleBase"] = queryMetaDF["ruleName"].apply(get_rule_base)

queryMetaDF["_ruleKey"] = queryMetaDF["ruleBase"].astype("object")
knownMetaDF["_ruleKey"] = knownMetaDF["ruleBase"].astype("object")

# Create ruleBase first, then use it
queryMetaDF["ruleBase"] = queryMetaDF["ruleName"].apply(get_rule_base)

if "ruleBase" not in knownMetaDF.columns:
    raise KeyError("knownMetaDF does not contain ruleBase after merge. Check knownOriginalIndex alignment.")

queryMetaDF["_ruleKey"] = queryMetaDF["ruleBase"].astype("object")
knownMetaDF["_ruleKey"] = knownMetaDF["ruleBase"].astype("object")

knownIndicesByRuleBase = {
    ruleBase: idx.to_numpy()
    for ruleBase, idx in knownMetaDF.dropna(subset=["_ruleKey"]).groupby("_ruleKey").groups.items()
}

queryRuleBases = set(queryMetaDF["ruleBase"].dropna().astype(str))
knownRuleBases = set(knownMetaDF["ruleBase"].dropna().astype(str))

print("Source of known enzymetic data base:")
print(knownMetaDF["ruleBaseSource"].value_counts(dropna=False))
print("Reaction rules present in DORAnet reaction mechanism:", len(queryRuleBases))
print("Reaction rules present in known enzymetic data base:", len(knownRuleBases))
print("Overlapping base reaction rules:", len(queryRuleBases.intersection(knownRuleBases)))
print("Example overlaps:", sorted(queryRuleBases.intersection(knownRuleBases))[:20])

Source of known enzymetic data base:
ruleBaseSource
top_mapped_operator     45763
all_mapped_operators     6882
no_ruleBase              1418
Name: count, dtype: int64
Reaction rules present in DORAnet reaction mechanism: 31
Reaction rules present in known enzymetic data base: 687
Overlapping base reaction rules: 29
Example overlaps: ['rule0002', 'rule0003', 'rule0007', 'rule0010', 'rule0011', 'rule0016', 'rule0017', 'rule0028', 'rule0042', 'rule0043', 'rule0047', 'rule0048', 'rule0056', 'rule0061', 'rule0062', 'rule0070', 'rule0073', 'rule0074', 'rule0083', 'rule0085']


### Run `ruleBase` constrained RCMFP enzyme retrieval

For each query reaction (present in DORAnet reaction network) it searches the known natural enzyme reaction database, computes RCMFP Tanimoto similarity, keeps the best matches and stores the results in `matchedRCMFP_DF`. Final returned matches are ranked from highest to lowest RCMFP similarity.

In [43]:
# this function calculates Tanimoto similarity between one query fingerprint and many known enzyme reference fingerprints
def compute_tanimoto_similarity(queryPackedRow, queryPop, refPackedMatrix, refPopcounts, popcount8):
    inter = popcount8[np.bitwise_and(refPackedMatrix, queryPackedRow)].sum(axis=1).astype(np.float32); union = refPopcounts + queryPop - inter
    scores = np.zeros(len(refPopcounts), dtype=np.float32); valid = union > 0; scores[valid] = inter[valid] / union[valid]
    return scores

# this function retrieves the top enzyme reference matches for one query reaction
def retrieve_similar_reactions(queryIdx, topK=TOP_K, minSimilarity=MIN_SIMILARITY):
    queryRuleBase = queryMetaDF.loc[queryIdx, "_ruleKey"]
    candidateIdx, searchMode = (knownIndicesByRuleBase[queryRuleBase], "same_ruleBase") if pd.notna(queryRuleBase) and queryRuleBase in knownIndicesByRuleBase else (np.arange(len(knownMetaDF)), "full_database_fallback")
    qPacked, qPop = queryPacked[queryIdx], queryPopcounts[queryIdx]
    scores = np.maximum(compute_tanimoto_similarity(qPacked, qPop, knownPacked[candidateIdx], knownPopcounts[candidateIdx], popcount8), compute_tanimoto_similarity(qPacked, qPop, knownPackedReverse[candidateIdx], knownReversePopcounts[candidateIdx], popcount8))
    validLocal = np.where(scores > minSimilarity)[0]
    if len(validLocal) == 0: return []
    topLocal = validLocal[np.argpartition(-scores[validLocal], topK - 1)[:topK]] if len(validLocal) > topK else validLocal
    topLocal = topLocal[np.argsort(-scores[topLocal])]
    return [{"queryRowId": int(queryIdx), "rank": rank, "knownRowId": int(candidateIdx[localIdx]), "rcmfpSimilarity": float(scores[localIdx]), "searchMode": searchMode, "numCandidatesSearched": int(len(candidateIdx))} for rank, localIdx in enumerate(topLocal, start=1)]

query_FP_Matrix, naturalEnzyme_fingerprint_matrix, naturalEnzyme_fingerprint_matrixReverse = query_FP_Matrix.astype(bool), naturalEnzyme_fingerprint_matrix.astype(bool), naturalEnzyme_fingerprint_matrixReverse.astype(bool)
queryPacked, knownPacked, knownPackedReverse = np.packbits(query_FP_Matrix.astype(np.uint8), axis=1), np.packbits(naturalEnzyme_fingerprint_matrix.astype(np.uint8), axis=1), np.packbits(naturalEnzyme_fingerprint_matrixReverse.astype(np.uint8), axis=1)
queryPopcounts, knownPopcounts, knownReversePopcounts = query_FP_Matrix.sum(axis=1).astype(np.int32), naturalEnzyme_fingerprint_matrix.sum(axis=1).astype(np.int32), naturalEnzyme_fingerprint_matrixReverse.sum(axis=1).astype(np.int32)
popcount8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

allHitRecords = []
for queryIdx in tqdm(range(len(queryMetaDF)), desc="RCMFP enzyme retrieval"): allHitRecords.extend(retrieve_similar_reactions(queryIdx))
matchedRCMFP_DF = pd.DataFrame(allHitRecords)
print("Total hits:", len(matchedRCMFP_DF)); print(matchedRCMFP_DF["searchMode"].value_counts(dropna=False))
matchedRCMFP_DF.head()

RCMFP enzyme retrieval: 100%|█████████████████████████████████████████████████████████████████████████| 4733/4733 [00:28<00:00, 164.39it/s]


Total hits: 86771
searchMode
same_ruleBase             85631
full_database_fallback     1140
Name: count, dtype: int64


,queryRowId,rank,knownRowId,rcmfpSimilarity,searchMode,numCandidatesSearched
0,0,1,6153,0.258929,same_ruleBase,41
1,0,2,2936,0.258160,same_ruleBase,41
2,0,3,42275,0.258160,same_ruleBase,41
3,0,4,2940,0.258160,same_ruleBase,41
4,0,5,42137,0.258160,same_ruleBase,41


### Build full summary, filter Moderate/Strong, deduplicate and save outputs

In [44]:
def print_section(title):
    print("\n" + "-" * 80)
    print(title)
    print("-" * 80)

def print_saved_file(label, path):
    print(f"{label}: {path}")

def classify_precedent(sim):
    if pd.isna(sim): return "No natural precedent retrieved"
    if sim >= 0.60: return "Strong natural enzyme precedent"
    if sim >= 0.35: return "Moderate natural enzyme precedent"
    if sim > 0: return "Weak precedent; likely enzyme-engineering risk"
    return "No useful RCMFP precedent"

queryDisplayCols = [c for c in ["queryRowId", "reactionString", "ruleName", "ruleBase", "reactionType", "queryReaction", "queryMappedReaction", "candidateUniProtRaw", "numCandidateUniProt", "rcmfpMappingStatus", "rcmfpStatus", "feasibilityScore_rule1", "feasibilityLabel_rule1", "feasibilityScore_rule2", "feasibilityLabel_rule2", "feasibilityScore_rule3", "feasibilityLabel_rule3", "feasibilityScore_rule4", "feasibilityLabel_rule4"] if c in queryMetaDF.columns]
knownDisplayCols = [c for c in ["knownRowId", "knownOriginalIndex", "ruleBase", "ruleBaseSource", "rxn_idx", "mapped", "unmapped", "orig_rxn_text", "rule", "source", "quality", "natural", "organism", "protein_refs", "protein_db", "ec_num", "top_mapped_operator", "all_mapped_operators"] if c in knownMetaDF.columns]

matchedRCMFP_DF = matchedRCMFP_DF.merge(queryMetaDF[queryDisplayCols], on="queryRowId", how="left", suffixes=("", "_query")).merge(knownMetaDF[knownDisplayCols], on="knownRowId", how="left", suffixes=("", "_known")).sort_values(["queryRowId", "rank"]).reset_index(drop=True)
matchedRCMFP_DF.head()

,queryRowId,rank,knownRowId,rcmfpSimilarity,searchMode,numCandidatesSearched,reactionString,ruleName,ruleBase,reactionType,...,rule,source,quality,natural,organism,protein_refs,protein_db,ec_num,top_mapped_operator,all_mapped_operators
0,0,1,6153,0.258929,same_ruleBase,41,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,rule0165,Enzymatic,...,[#6:1].[#6:2]1:[#6:3]:[#6:4]:[#6:5]:[#7+:6]:[#...,direct,0.008197,0.0,Ovis aries,[],None,1.2.1.3,rule0165,[rule0165]
1,0,2,2936,0.258160,same_ruleBase,41,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,rule0165,Enzymatic,...,[#6:1].[#6:2]1:[#6:3]:[#6:4]:[#6:5]:[#7+:6]:[#...,direct,1.000000,0.0,Pseudomonas aeruginosa,['O86422'],swissprot,1.1.1.22,None,[rule0165]
2,0,3,42275,0.258160,same_ruleBase,41,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,rule0165,Enzymatic,...,[#6:1].[#6:2]1:[#6:3]:[#6:4]:[#6:5]:[#7+:6]:[#...,direct,1.000000,NaN,None,[],None,1.1.1.22,None,[rule0165]
3,0,4,2940,0.258160,same_ruleBase,41,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,rule0165,Enzymatic,...,[#6:1].[#6:2]1:[#6:3]:[#6:4]:[#6:5]:[#7+:6]:[#...,direct,1.000000,0.0,Bos taurus,[],None,1.1.1.22,rule0165,[rule0165]
4,0,5,42137,0.258160,same_ruleBase,41,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,rule0165,Enzymatic,...,[#6:1].[#6:2]1:[#6:3]:[#6:4]:[#6:5]:[#7+:6]:[#...,direct,1.000000,NaN,None,[],None,1.1.1.186,rule0165,[rule0165]


### Marge information of natural enzyme reference to `reactionDF_wUniprotID`

In [45]:
# User options
keyCol = "reactionString"
scoreCol = "rcmfpSimilarity"
keepMode = "best"   # "best" or "all"
bestAscending = False  # False => highest score is best, True => lowest is best

sourceDF = matchedRCMFP_DF.copy()

if keepMode == "best":
    sourceToMerge = (
        sourceDF
        .sort_values(scoreCol, ascending=bestAscending, na_position="last")
        .drop_duplicates(subset=[keyCol], keep="first")
    )
elif keepMode == "all":
    sourceToMerge = sourceDF
else:
    raise ValueError("keepMode must be 'best' or 'all'")

# only add columns that don't already exist in target
newCols = [c for c in sourceToMerge.columns if c not in reactionDF_wUniprotID.columns and c != keyCol]

reactionDF_wUniprotID = reactionDF_wUniprotID.merge(
    sourceToMerge[[keyCol] + newCols],
    on=keyCol,
    how="inner"   # keep only rows present in source
)

print(f"Mode: {keepMode}")
print(f"Rows after merge: {len(reactionDF_wUniprotID):,}")
print(f"Added columns: {len(newCols)}")
reactionDF_wUniprotID

Mode: best
Rows after merge: 4,733
Added columns: 28


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,rule,source,quality,natural,organism,protein_refs,protein_db,ec_num,top_mapped_operator,all_mapped_operators
0,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,No_Thermo,"(1, 1, 1, 1)","(1, 1, 1)",Enzymatic,4,3,...,[#6:1].[#6:2]1:[#6:3]:[#6:4]:[#6:5]:[#7+:6]:[#...,direct,0.008197,0.0,Ovis aries,[],None,1.2.1.3,rule0165,[rule0165]
1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,[#6:1]-[#8:2].[#6:3]1=[#6:4]-[#7:5]-[#6:6]=[#6...,direct,0.333333,NaN,None,[],None,1.14.13.246,rule0073,"[rule0073, rule0491]"
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,[#6:1]-[#16:2].[#8:3]>>[#16:2].[#6:1]-[#8:3],direct reversed,1.000000,1.0,Cavia porcellus,[],None,2.3.1.44,rule0061,"[rule0021, rule0059, rule0061, rule0089, rule0..."
3,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,[#6:1]-[#16:2].[#8:3]>>[#16:2].[#6:1]-[#8:3],direct reversed,1.000000,1.0,Cavia porcellus,[],None,2.3.1.44,rule0061,"[rule0021, rule0059, rule0061, rule0089, rule0..."
4,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,[#6:1]-[#16:2].[#8:3]>>[#16:2].[#6:1]-[#8:3],direct reversed,1.000000,1.0,Cavia porcellus,[],None,2.3.1.44,rule0061,"[rule0021, rule0059, rule0061, rule0089, rule0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4728,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,[#6:1]-[#8:2].[#6:3]1=[#6:4]-[#7:5]-[#6:6]=[#6...,direct,0.333333,NaN,None,[],None,1.14.13.246,rule0073,"[rule0073, rule0491]"
4729,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,[#6:1]-[#16:2].[#8:3]>>[#16:2].[#6:1]-[#8:3],direct reversed,1.000000,1.0,Cavia porcellus,[],None,2.3.1.44,rule0061,"[rule0021, rule0059, rule0061, rule0089, rule0..."
4730,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,[#6:1]-[#16:2].[#8:3]>>[#16:2].[#6:1]-[#8:3],direct reversed,1.000000,1.0,Cavia porcellus,[],None,2.3.1.44,rule0061,"[rule0021, rule0059, rule0061, rule0089, rule0..."
4731,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,[#6:1]-[#16:2].[#8:3]>>[#16:2].[#6:1]-[#8:3],direct reversed,1.000000,1.0,Cavia porcellus,[],None,2.3.1.44,rule0061,"[rule0021, rule0059, rule0061, rule0089, rule0..."


### Find best `UniProt ID` per DORANet reaction rule

In [47]:
uniprotPattern = re.compile(
    r"([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z0-9]{3}[0-9]){1,2})"
)

def extractUniProtID(x):
    if pd.isna(x):
        return []
    ids = uniprotPattern.findall(str(x))
    seen, out = set(), []
    for uid in ids:
        if uid not in seen:
            out.append(uid)
            seen.add(uid)
    return out


tmp = matchedRCMFP_DF.copy()

tmp["knownProteinRefList"] = tmp["protein_refs"].apply(extractUniProtID)

tmp = tmp[
    tmp["knownProteinRefList"].apply(len) > 0
].copy()

tmp = tmp.explode("knownProteinRefList").rename(
    columns={"knownProteinRefList": "rcmfpSupportedUniProt"}
)

tmp["rcmfpSimilarity"] = pd.to_numeric(
    tmp["rcmfpSimilarity"],
    errors="coerce"
)

tmp["rank"] = pd.to_numeric(
    tmp["rank"],
    errors="coerce"
)

ruleRcmfpUniProtScoreDF = (
    tmp
    .groupby(["ruleName", "ruleBase", "rcmfpSupportedUniProt"])
    .agg(
        numSupportingHits=("knownRowId", "nunique"),
        bestRCMFPSimilarity=("rcmfpSimilarity", "max"),
        meanRCMFPSimilarity=("rcmfpSimilarity", "mean"),
        bestRank=("rank", "min"),
        exampleKnownRowId=("knownRowId", "first"),
        exampleKnownReaction=("orig_rxn_text", "first"),
        exampleKnownOrganism=("organism", "first"),
        exampleKnownEcNumber=("ec_num", "first"),
        exampleProteinRefs=("protein_refs", "first"),
    )
    .reset_index()
)

ruleRcmfpUniProtScoreDF["rcmfpProteinRefScore"] = (
    ruleRcmfpUniProtScoreDF["bestRCMFPSimilarity"] * 100
    + np.log1p(ruleRcmfpUniProtScoreDF["numSupportingHits"]) * 10
    - ruleRcmfpUniProtScoreDF["bestRank"] * 0.1
)

ruleRcmfpUniProtScoreDF = ruleRcmfpUniProtScoreDF.sort_values(
    [
        "ruleName",
        "rcmfpProteinRefScore",
        "bestRCMFPSimilarity",
        "numSupportingHits",
        "bestRank",
    ],
    ascending=[True, False, False, False, True]
)

bestRCMFPUniProtByRuleDF = (
    ruleRcmfpUniProtScoreDF
    .groupby("ruleName")
    .head(1)
    .reset_index(drop=True)
)

bestRCMFPUniProtByRuleDF[
    [
        "ruleName",
        "rcmfpSupportedUniProt",
        "rcmfpProteinRefScore",
        "bestRCMFPSimilarity",
        "meanRCMFPSimilarity",
        "numSupportingHits",
        "bestRank",
        "exampleKnownOrganism",
        "exampleKnownEcNumber",
        "exampleKnownReaction",
    ]
].head()

,ruleName,rcmfpSupportedUniProt,rcmfpProteinRefScore,bestRCMFPSimilarity,meanRCMFPSimilarity,numSupportingHits,bestRank,exampleKnownOrganism,exampleKnownEcNumber,exampleKnownReaction
0,rule0003_171,B2KJ46,31.739969,0.211538,0.211538,2,4,Candida parapsilosis,1.1.1.1,2-hydroxyacetophenone + NADPH + H+ = (S)-1-phe...
1,rule0003_176,A0A0H4SN47,30.228112,0.204420,0.201273,2,12,Ogataea glucozyma,1.1.1.320,1-(2-nitrophenyl)ethanone + NADPH + H+ = (1S)-...
2,rule0007_161,P27169,31.731694,0.179688,0.167985,3,1,Homo sapiens,3.1.1.2,"2,2,2-trifluoroethyl acetate + H2O = 2,2,2-tri..."
3,rule0007_174,B1P195,23.828464,0.187970,0.180622,1,19,Bifidobacterium breve,3.2.1.37,ginsenoside Ra2 + H2O = ginsenoside Rc + D-xylose
4,rule0007_193,Q8NKS0,24.876584,0.180451,0.180451,1,1,Pyrobaculum calidifontis,3.1.1.1,sec-butyl acetate + H2O = sec-butanol + acetate


### Merge back `RCMFP-supported UniProt ID` with `reactionDF_wUniprotID`

In [48]:
# Columns to keep + rename map
rename_map = {
    "rcmfpSupportedUniProt": "bestRCMFP_UniProtID",
    "rcmfpProteinRefScore": "bestRCMFP_UniProtScore",
    "bestRCMFPSimilarity": "bestRCMFP_SimilarityScore",
    "meanRCMFPSimilarity": "meanRCMFP_Similarity_byRule",
    "numSupportingHits": "numRCMFP_SupportingHits_byRule",
    "bestRank": "bestRCMFPRank_byRule",
    "exampleKnownRowId": "bestRCMFP_RowId",
    "exampleKnownReaction": "bestRCMFP_Reaction",
    "exampleKnownOrganism": "bestRCMFP_Organism",
    "exampleKnownEcNumber": "bestRCMFP_ECnumber",
    "exampleProteinRefs": "bestRCMFP_ProteinRefs",
}

bestRCMFPForMergeDF = (
    bestRCMFPUniProtByRuleDF[["ruleName", *rename_map]]
    .rename(columns=rename_map)
)

reactionDF_wBestRCMFP_Similarity = (
    reactionDF_wUniprotID
    .merge(bestRCMFPForMergeDF, on="ruleName", how="left")
    .assign(
        hasbestRCMFP_UniProtID=lambda df: (
            df["bestRCMFP_UniProtID"].fillna("").astype(str).str.len().gt(0)
        )
    )
)

print("reactionDF_wBestRCMFP_Similarity created")
print(f"Rows in reactionDF_wUniprotID              : {len(reactionDF_wUniprotID):,}")
print(f"Rows in reactionDF_wBestRCMFP_Similarity   : {len(reactionDF_wBestRCMFP_Similarity):,}")
print(f"Rows with best RCMFP UniProt               : {reactionDF_wBestRCMFP_Similarity['hasbestRCMFP_UniProtID'].sum():,}")
print(f"Unique best RCMFP UniProt IDs              : {reactionDF_wBestRCMFP_Similarity['bestRCMFP_UniProtID'].nunique():,}")

reactionDF_wBestRCMFP_Similarity.head()

reactionDF_wBestRCMFP_Similarity created
Rows in reactionDF_wUniprotID              : 4,733
Rows in reactionDF_wBestRCMFP_Similarity   : 4,733
Rows with best RCMFP UniProt               : 3,513
Unique best RCMFP UniProt IDs              : 35


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,bestRCMFP_SimilarityScore,meanRCMFP_Similarity_byRule,numRCMFP_SupportingHits_byRule,bestRCMFPRank_byRule,bestRCMFP_RowId,bestRCMFP_Reaction,bestRCMFP_Organism,bestRCMFP_ECnumber,bestRCMFP_ProteinRefs,hasbestRCMFP_UniProtID
0,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,No_Thermo,"(1, 1, 1, 1)","(1, 1, 1)",Enzymatic,4,3,...,0.207386,0.199784,6.0,4.0,2280.0,1-octadecanol + 2 NAD+ + H2O = octadecanoic ac...,Geobacillus thermodenitrificans,1.1.1.192,['A4IP64'],True
1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.101695,0.095830,1.0,2.0,11708.0,acetyl-CoA + ethanol = CoA + ethyl acetate {r},Wickerhamomyces anomalus,2.3.1.268,['A0A1E3P8S6'],True
3,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.101695,0.095830,1.0,2.0,11708.0,acetyl-CoA + ethanol = CoA + ethyl acetate {r},Wickerhamomyces anomalus,2.3.1.268,['A0A1E3P8S6'],True
4,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,0.101695,0.095830,1.0,2.0,11708.0,acetyl-CoA + ethanol = CoA + ethyl acetate {r},Wickerhamomyces anomalus,2.3.1.268,['A0A1E3P8S6'],True


In [49]:
reactionDF_wBestRCMFP_Similarity = reactionDF_wBestRCMFP_Similarity.drop(
    columns=[
        'reactantSetCanonical', 'productSetCanonical',
        'rxn_str', 'feasibilityScore_rule1', 'feasibilityLabel_rule1',
        'feasibilityLabel_rule2', 'feasibilityScore_rule3', 'feasibilityLabel_rule3',
        'feasibilityScore_rule4', 'feasibilityLabel_rule4', 'ruleReactants',
        'ruleProducts', 'ruleSMARTS', 'hasRuleLookup', 'queryRowId', 'rank',
        'knownRowId', 'rcmfpSimilarity', 'searchMode', 'numCandidatesSearched',
        'ruleBase', 'queryReaction', 'queryMappedReaction','rcmfpMappingStatus', 'rcmfpStatus', 'knownOriginalIndex',
        'ruleBase_known', 'ruleBaseSource', 'rxn_idx', 'mapped', 'unmapped','orig_rxn_text', 'rule', 'source', 'quality', 'natural', 
        'organism','protein_refs', 'protein_db', 'ec_num', 'top_mapped_operator',
        'all_mapped_operators','bestRCMFP_UniProtScore', 'meanRCMFP_Similarity_byRule','numRCMFP_SupportingHits_byRule',      
        'bestRCMFPRank_byRule','bestRCMFP_RowId', 'bestRCMFP_Reaction', 'bestRCMFP_ProteinRefs', 'hasbestRCMFP_UniProtID'               
    ],
    errors='ignore'  
)

reactionDF_wBestRCMFP_Similarity

,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,starterSMILES,targetSMILES,connectionMoleculesByStep,feasibilityScore_rule2,candidateUniProtRaw,numCandidateUniProt,bestRCMFP_UniProtID,bestRCMFP_SimilarityScore,bestRCMFP_Organism,bestRCMFP_ECnumber
0,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,No_Thermo,"(1, 1, 1, 1)","(1, 1, 1)",Enzymatic,4,3,...,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)CC(=O)O,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,0.967761,A0A0J9X7D2;A1BPP9;A4IP64;A4ISB9;B0S9F2;B0SS41;...,228,A4IP64,0.207386,Geobacillus thermodenitrificans,1.1.1.192
1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,0.738403,E3VWJ1;F1T282;F1T283;I1TEM3;O94142;Q16850;Q2MJ...,11,NaN,NaN,NaN,NaN
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,0.995089,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,A0A1E3P8S6,0.101695,Wickerhamomyces anomalus,2.3.1.268
3,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,0.995089,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,A0A1E3P8S6,0.101695,Wickerhamomyces anomalus,2.3.1.268
4,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,O | CC(=O)OCC1OC(n2cnc3c(=O)[nH]cnc32)C(O)C1O,0.995089,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,A0A1E3P8S6,0.101695,Wickerhamomyces anomalus,2.3.1.268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4728,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,NC(=O)c1ccc[n+](C2OC(COP(=O)(O)OP(=O)(O)OCC3OC...,0.738403,E3VWJ1;F1T282;F1T283;I1TEM3;O94142;Q16850;Q2MJ...,11,NaN,NaN,NaN,NaN
4729,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,NC(=O)c1ccc[n+](C2OC(COP(=O)(O)OP(=O)(O)OCC3OC...,0.995089,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,A0A1E3P8S6,0.101695,Wickerhamomyces anomalus,2.3.1.268
4730,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,0.995089,A0A0B4VSM7;A0MPB1;A1XWY7;B1PBV3;B5KRF3;B5KRF5;...,46,A0A1E3P8S6,0.101695,Wickerhamomyces ano

## 5. Query UniProt for EC numbers, protein names, organisms and sequences

```text
DORAnet rule ID → UniProt accession → EC number/protein sequence
```

### Summarize unique DORAnet reaction rules with `UniProt ID`

In [50]:
tmp = reactionDF_wBestRCMFP_Similarity.copy()

tmp["prioritizedUniProt"] = (
    tmp["bestRCMFP_UniProtID"].fillna("").astype(str).str.strip()
)
fallback_mask = tmp["prioritizedUniProt"].eq("")
tmp.loc[fallback_mask, "prioritizedUniProt"] = (
    tmp.loc[fallback_mask, "candidateUniProtRaw"].fillna("").astype(str).str.strip()
)

usedRuleSummaryDF = (
    tmp
    .groupby(["ruleName", "reactionType", "prioritizedUniProt"], dropna=False)
    .agg(
        numReactions=("reactionString", "count"),
        exampleReaction=("reactionString", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)

usedRuleSummaryDF

,ruleName,reactionType,prioritizedUniProt,numReactions,exampleReaction
44,rule0165_2,Enzymatic,A4IP64,649,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...
20,rule0017_16,Enzymatic,A0B8M6;A0KGH8;A0KU87;A0L4K6;A0LHG0;A0M5L6;A0RM...,592,O=P(O)(O)O.O=c1[nH]cnc2c1ncn2[C@@H]1OC(CO)[C@@...
31,rule0062_17,Enzymatic,Q4WZ64,553,CC(=O)O.CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)...
35,rule0073_5,Enzymatic,E3VWJ1;F1T282;F1T283;I1TEM3;O94142;Q16850;Q2MJ...,511,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...
8,rule0007_198,Enzymatic,Q86W56,452,CC(=O)O[C@@H]1[C@H](O)[C@@H](CO)O[C@]1(C)n1cnc...
30,rule0061_4,Enzymatic,A0A1E3P8S6,330,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...
4,rule0007_161,Enzymatic,P27169,239,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...
48,rule0491_2,Enzymatic,Q2KQ78,238,CC(O)O[C@@H]1[C@H]2CO[C@@]1(O)[C@H](n1cnc3c(=O...
32,rule0062_18,Enzymatic,P77791,154,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...
33,rule0062_19,Enzymatic,Q5Y9C7,153,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...


### Fetch `UniProt` metadata

In [51]:
def chunkList(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


def cleanUniProtAccession(x):
    if pd.isna(x):
        return None

    x = str(x).strip()
    x = x.replace("UniProtKB:", "").replace("UniProt:", "").replace("uniprot:", "")
    x = x.replace("sp|", "").replace("tr|", "")

    # Handles strings like sp|P77791|NAME
    if "|" in x:
        parts = [p.strip() for p in x.split("|") if p.strip()]
        for p in parts:
            if isValidUniProtAccession(p):
                return p

    x = x.split()[0].split("-")[0].strip()
    return x if x else None


uniprotAccessionPattern = re.compile(
    r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z0-9]{3}[0-9]){1,2})$"
)


def isValidUniProtAccession(x):
    return False if pd.isna(x) else bool(
        uniprotAccessionPattern.match(str(x).strip())
    )


def fetchUniProtMetadata(accessionList, chunkSize=20, sleepSeconds=0.5):
    fields = [
        "accession",
        "reviewed",
        "id",
        "protein_name",
        "gene_names",
        "organism_name",
        "organism_id",
        "ec",
        "length",
        "sequence",
    ]

    allDf = []
    errors = 0

    for chunk in tqdm(list(chunkList(accessionList, chunkSize)), desc="Querying UniProt"):
        q = " OR ".join([f"(accession_id:{a})" for a in chunk])

        try:
            r = requests.get(
                "https://rest.uniprot.org/uniprotkb/search",
                params={
                    "query": q,
                    "format": "tsv",
                    "fields": ",".join(fields),
                    "size": chunkSize,
                },
                timeout=120,
            )

            if r.status_code != 200:
                errors += 1
            else:
                df = pd.read_csv(StringIO(r.text), sep="\t")

                if len(df) == 0:
                    errors += 1
                else:
                    allDf.append(df)

        except Exception:
            errors += 1

        time.sleep(sleepSeconds)

    out = (
        pd.concat(allDf, ignore_index=True).drop_duplicates()
        if allDf
        else pd.DataFrame()
    )

    return out, errors


# -----------------------------------------------
# Filter best RCMFP UniProt IDs
# -----------------------------------------------
reactionDF_wBestRCMFP_Similarity = reactionDF_wBestRCMFP_Similarity.copy()

reactionDF_wBestRCMFP_Similarity["bestRCMFP_UniProtID_clean"] = (
    reactionDF_wBestRCMFP_Similarity["bestRCMFP_UniProtID"]
    .apply(cleanUniProtAccession)
)

reactionDF_wBestRCMFP_Similarity["isValidBestRCMFP_UniProtID"] = (
    reactionDF_wBestRCMFP_Similarity["bestRCMFP_UniProtID_clean"]
    .apply(isValidUniProtAccession)
)

uniqueAccessionList = sorted(
    reactionDF_wBestRCMFP_Similarity.loc[
        reactionDF_wBestRCMFP_Similarity["isValidBestRCMFP_UniProtID"],
        "bestRCMFP_UniProtID_clean",
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(f"Unique valid best RCMFP UniProt IDs: {len(uniqueAccessionList):,}")

if not uniqueAccessionList:
    raise RuntimeError("No valid best RCMFP UniProt IDs found.")


# -----------------------------------------------
# Query UniProt
# -----------------------------------------------
uniprotMetadataRawDF, failedChunks = fetchUniProtMetadata(
    uniqueAccessionList,
    chunkSize=20,
    sleepSeconds=0.5,
)

totalChunks = int(np.ceil(len(uniqueAccessionList) / 20))
successChunks = totalChunks - failedChunks

print(f"Successful query chunks: {successChunks}")
print(f"Failed query chunks    : {failedChunks}")


# -----------------------------------------------
# Standardize UniProt columns
# -----------------------------------------------
renameMap = {
    "Entry": "uniprotAccession",
    "Reviewed": "uniprotReviewed",
    "Entry Name": "entryName",
    "Protein names": "proteinName",
    "Gene Names": "geneNames",
    "Organism": "organism",
    "Organism (ID)": "organismTaxId",
    "EC number": "ecNumber",
    "Length": "proteinLengthAa",
    "Sequence": "proteinSequence",
}

uniprotMetadataDF = uniprotMetadataRawDF.rename(columns=renameMap)

requiredCols = [
    "uniprotAccession",
    "uniprotReviewed",
    "entryName",
    "proteinName",
    "geneNames",
    "organism",
    "organismTaxId",
    "ecNumber",
    "proteinLengthAa",
    "proteinSequence",
]

for c in requiredCols:
    if c not in uniprotMetadataDF.columns:
        uniprotMetadataDF[c] = np.nan

uniprotMetadataDF = uniprotMetadataDF[requiredCols].copy()


# -----------------------------------------------
# Merge UniProt metadata back to reactionDF_wBestRCMFP_Similarity
# -----------------------------------------------
reactionDF_wBestRCMFP_Similarity = reactionDF_wBestRCMFP_Similarity.rename(
    columns={"bestRCMFP_UniProtID_clean": "uniprotAccession"}
)

reactionDF_wBestRCMFP_wUniProtMetadata = reactionDF_wBestRCMFP_Similarity.merge(
    uniprotMetadataDF,
    on="uniprotAccession",
    how="left",
    suffixes=("", "_uniprot"),
)

reactionDF_wBestRCMFP_wUniProtMetadata["hasUniProtMetadata"] = (
    reactionDF_wBestRCMFP_wUniProtMetadata["proteinName"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

reactionDF_wBestRCMFP_wUniProtMetadata["hasProteinSequence"] = (
    reactionDF_wBestRCMFP_wUniProtMetadata["proteinSequence"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

print("reactionDF_wBestRCMFP_wUniProtMetadata created")
print(f"Rows total             : {len(reactionDF_wBestRCMFP_wUniProtMetadata):,}")
print(f"Rows with metadata     : {reactionDF_wBestRCMFP_wUniProtMetadata['hasUniProtMetadata'].sum():,}")
print(f"Rows with protein seq  : {reactionDF_wBestRCMFP_wUniProtMetadata['hasProteinSequence'].sum():,}")

reactionDF_wBestRCMFP_wUniProtMetadata.head()

Unique valid best RCMFP UniProt IDs: 35


Querying UniProt: 100%|██████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.42s/it]

Successful query chunks: 2
Failed query chunks    : 0
reactionDF_wBestRCMFP_wUniProtMetadata created
Rows total             : 4,733
Rows with metadata     : 3,513
Rows with protein seq  : 3,483


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,entryName,proteinName,geneNames,organism,organismTaxId,ecNumber,proteinLengthAa,proteinSequence,hasUniProtMetadata,hasProteinSequence
0,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,No_Thermo,"(1, 1, 1, 1)","(1, 1, 1)",Enzymatic,4,3,...,ADH1_GEOTN,Long-chain-alcohol dehydrogenase 1 (EC 1.1.1.1...,adh1 GTNG_1754,Geobacillus thermodenitrificans (strain NG80-2),420246.0,1.1.1.192; 1.1.1.6,395.0,MSVARIVFPPLSHVGWGALDQLVPEVKRLGAKHILVITDPMLVKIG...,True,True
1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,EAT1_WICAA,Ethanol acetyltransferase 1 (EC 2.3.1.268) (Ac...,EAT1 WICANDRAFT_27004,Wickerhamomyces anomalus (strain ATCC 58044 / ...,683960.0,2.3.1.268; 3.1.1.-; 3.1.2.1,391.0,MFFTKVLNNQVANGLKQLPVHKRVQMAYDLHIPNKTVNPNLNIRSH...,True,True
3,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,EAT1_WICAA,Ethanol acetyltransferase 1 (EC 2.3.1.268) (Ac...,EAT1 WICANDRAFT_27004,Wickerhamomyces anomalus (strain ATCC 58044 / ...,683960.0,2.3.1.268; 3.1.1.-; 3.1.2.1,391.0,MFFTKVLNNQVANGLKQLPVHKRVQMAYDLHIPNKTVNPNLNIRSH...,True,True
4,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,EAT1_WICAA,Ethanol acetyltransferase 1 (EC 2.3.1.268) (Ac...,EAT1 WICANDRAFT_27004,Wickerhamomyces anomalus (strain ATCC 58044 / ...,683960.0,2.3.1.268; 3.1.1.-; 3.1.2.1,391.0,MFFTKVLNNQVANGLKQLPVHKRVQMAYDLHIPNKTVNPNLNIRSH...,True,True


## 6. Fetch `DNA sequences` from `GenBank`

In [52]:
# -----------------------------------------------
# settings
# -----------------------------------------------

NCBI_EMAIL = "sghosh6@lbl.gov"
NCBI_API_KEY = None
NCBI_TOOL = "DORAnet_GenBank_CDS_Fetch"

ncbiSleepSeconds = 0.40 if NCBI_API_KEY is None else 0.12
uniprotSleepSeconds = 0.15

# For testing, set 20 or 50. For full run, keep None.
maxUniProtToFetch = None


# -----------------------------------------------
# UniProt column setup
# -----------------------------------------------
if "reactionDF_wBestRCMFP_wUniProtMetadata" not in globals():
    raise RuntimeError("reactionDF_wBestRCMFP_wUniProtMetadata is not defined.")

reactionDF_wBestRCMFP_wUniProtMetadata = reactionDF_wBestRCMFP_wUniProtMetadata.copy()

# Prefer existing uniprotAccession column.
# Otherwise use bestRCMFP_UniProtID.
if "uniprotAccession" not in reactionDF_wBestRCMFP_wUniProtMetadata.columns:
    if "bestRCMFP_UniProtID" in reactionDF_wBestRCMFP_wUniProtMetadata.columns:
        reactionDF_wBestRCMFP_wUniProtMetadata["uniprotAccession"] = (
            reactionDF_wBestRCMFP_wUniProtMetadata["bestRCMFP_UniProtID"]
        )
    else:
        raise ValueError("Need either 'uniprotAccession' or 'bestRCMFP_UniProtID' column.")


# -----------------------------------------------
# Helper functions
# -----------------------------------------------
def dedupeKeepOrder(valueList):
    seen = set()
    out = []

    for value in valueList:
        if value is None:
            continue

        value = str(value).strip()

        if value and value not in seen:
            out.append(value)
            seen.add(value)

    return out


def cleanUniProtAccession(x):
    if pd.isna(x):
        return None

    x = str(x).strip()
    x = (
        x.replace("UniProtKB:", "")
        .replace("UniProt:", "")
        .replace("uniprot:", "")
        .replace("sp|", "")
        .replace("tr|", "")
    )

    if "|" in x:
        parts = [p.strip() for p in x.split("|") if p.strip()]
        for p in parts:
            if isValidUniProtAccession(p):
                return p

    x = x.split()[0].split("-")[0].strip()

    return x if x else None


uniprotAccessionPattern = re.compile(
    r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z0-9]{3}[0-9]){1,2})$"
)


def isValidUniProtAccession(x):
    return False if pd.isna(x) else bool(
        uniprotAccessionPattern.match(str(x).strip())
    )


def parseFastaRecords(fastaText):
    records = []
    header = None
    seqLines = []

    for line in str(fastaText).splitlines():
        line = line.strip()

        if not line:
            continue

        if line.startswith(">"):
            if header is not None:
                records.append({
                    "header": header,
                    "sequence": "".join(seqLines),
                })

            header = line[1:]
            seqLines = []

        else:
            seqLines.append(line)

    if header is not None:
        records.append({
            "header": header,
            "sequence": "".join(seqLines),
        })

    return records


def cleanDnaSequence(seq):
    seq = str(seq).upper()
    seq = re.sub(r"[^ACGTN]", "", seq)
    return seq


def fetchUniProtCrossRefs(uniprotAccession, sleepSeconds=0.15):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprotAccession}.json"

    result = {
        "uniprotAccession": uniprotAccession,
        "refseqProteinIds": [],
        "emblProteinIds": [],
        "nucleotideIds": [],
        "rawCrossRefSummary": "",
        "uniprotCrossRefStatus": "not_queried",
        "uniprotCrossRefError": "",
    }

    try:
        response = requests.get(url, timeout=60)

        if response.status_code != 200:
            result["uniprotCrossRefStatus"] = "failed"
            result["uniprotCrossRefError"] = f"{response.status_code}: {response.text[:300]}"
            time.sleep(sleepSeconds)
            return result

        entryJson = response.json()
        crossRefs = entryJson.get("uniProtKBCrossReferences", [])

        refseqProteinIds = []
        emblProteinIds = []
        nucleotideIds = []
        rawSummaryParts = []

        for xref in crossRefs:
            databaseName = str(xref.get("database", "")).strip()
            xrefId = str(xref.get("id", "")).strip()
            properties = xref.get("properties", [])

            propertyDict = {}

            for prop in properties:
                key = str(prop.get("key", "")).strip()
                value = str(prop.get("value", "")).strip()

                if key and value:
                    propertyDict.setdefault(key, []).append(value)

            if databaseName in ["RefSeq", "EMBL", "GenBank", "DDBJ"]:
                rawSummaryParts.append(f"{databaseName}:{xrefId}")

            if databaseName == "RefSeq" and xrefId:
                refseqProteinIds.append(xrefId)

            if databaseName in ["EMBL", "GenBank", "DDBJ"]:
                if xrefId:
                    nucleotideIds.append(xrefId)

                for key, values in propertyDict.items():
                    normalizedKey = (
                        key.lower()
                        .replace(" ", "")
                        .replace("_", "")
                        .replace("-", "")
                    )

                    if "proteinid" in normalizedKey or normalizedKey == "protein":
                        emblProteinIds.extend(values)

                    if "nucleotidesequenceid" in normalizedKey or "nucleotide" in normalizedKey:
                        nucleotideIds.extend(values)

        result["refseqProteinIds"] = dedupeKeepOrder(refseqProteinIds)
        result["emblProteinIds"] = dedupeKeepOrder(emblProteinIds)
        result["nucleotideIds"] = dedupeKeepOrder(nucleotideIds)
        result["rawCrossRefSummary"] = ";".join(rawSummaryParts)
        result["uniprotCrossRefStatus"] = "success"

    except Exception as exc:
        result["uniprotCrossRefStatus"] = "failed"
        result["uniprotCrossRefError"] = str(exc)

    time.sleep(sleepSeconds)

    return result


def fetchNcbiCdsFromProteinAccession(proteinAccession, sleepSeconds=0.40):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

    params = {
        "db": "protein",
        "id": proteinAccession,
        "rettype": "fasta_cds_na",
        "retmode": "text",
        "tool": NCBI_TOOL,
        "email": NCBI_EMAIL,
    }

    if NCBI_API_KEY is not None:
        params["api_key"] = NCBI_API_KEY

    result = {
        "proteinAccessionQueried": proteinAccession,
        "ncbiFetchStatus": "not_queried",
        "ncbiFetchError": "",
        "genbankCdsFastaHeader": "",
        "genbankCdsSequence": "",
        "genbankCdsLengthBp": np.nan,
        "numFastaRecordsReturned": 0,
    }

    try:
        response = requests.get(url, params=params, timeout=120)

        if response.status_code != 200:
            result["ncbiFetchStatus"] = "failed"
            result["ncbiFetchError"] = f"{response.status_code}: {response.text[:300]}"
            time.sleep(sleepSeconds)
            return result

        fastaRecords = parseFastaRecords(response.text.strip())

        if len(fastaRecords) == 0:
            result["ncbiFetchStatus"] = "no_cds_returned"
            result["ncbiFetchError"] = response.text[:300]
            time.sleep(sleepSeconds)
            return result

        firstRecord = fastaRecords[0]
        cleanSeq = cleanDnaSequence(firstRecord["sequence"])

        if len(cleanSeq) == 0:
            result["ncbiFetchStatus"] = "empty_cds_sequence"
            result["ncbiFetchError"] = "No valid DNA sequence parsed."
            time.sleep(sleepSeconds)
            return result

        result["ncbiFetchStatus"] = "success"
        result["genbankCdsFastaHeader"] = firstRecord["header"]
        result["genbankCdsSequence"] = cleanSeq
        result["genbankCdsLengthBp"] = len(cleanSeq)
        result["numFastaRecordsReturned"] = len(fastaRecords)

    except Exception as exc:
        result["ncbiFetchStatus"] = "failed"
        result["ncbiFetchError"] = str(exc)

    time.sleep(sleepSeconds)

    return result


def fetchBestGenBankCdsForUniProt(uniprotAccession):
    crossRefResult = fetchUniProtCrossRefs(
        uniprotAccession,
        sleepSeconds=uniprotSleepSeconds,
    )

    candidateProteinIds = dedupeKeepOrder(
        crossRefResult["refseqProteinIds"] + crossRefResult["emblProteinIds"]
    )

    output = {
        "uniprotAccession": uniprotAccession,
        "genbankProteinIdsFromUniProt": ";".join(candidateProteinIds),
        "genbankNucleotideIdsFromUniProt": ";".join(crossRefResult["nucleotideIds"]),
        "rawCrossRefSummary": crossRefResult["rawCrossRefSummary"],
        "uniprotCrossRefStatus": crossRefResult["uniprotCrossRefStatus"],
        "uniprotCrossRefError": crossRefResult["uniprotCrossRefError"],
        "genbankProteinAccessionUsed": "",
        "genbankCdsFastaHeader": "",
        "genbankCdsSequence": "",
        "genbankCdsLengthBp": np.nan,
        "numFastaRecordsReturned": 0,
        "genbankCdsFetchStatus": "not_attempted",
        "genbankCdsFetchError": "",
    }

    if len(candidateProteinIds) == 0:
        output["genbankCdsFetchStatus"] = "no_protein_crossref"
        output["genbankCdsFetchError"] = (
            "No RefSeq/EMBL/GenBank protein cross-reference found in UniProt JSON."
        )
        return output

    fetchErrors = []

    for proteinAccession in candidateProteinIds:
        cdsResult = fetchNcbiCdsFromProteinAccession(
            proteinAccession,
            sleepSeconds=ncbiSleepSeconds,
        )

        if cdsResult["ncbiFetchStatus"] == "success":
            output["genbankProteinAccessionUsed"] = proteinAccession
            output["genbankCdsFastaHeader"] = cdsResult["genbankCdsFastaHeader"]
            output["genbankCdsSequence"] = cdsResult["genbankCdsSequence"]
            output["genbankCdsLengthBp"] = cdsResult["genbankCdsLengthBp"]
            output["numFastaRecordsReturned"] = cdsResult["numFastaRecordsReturned"]
            output["genbankCdsFetchStatus"] = "success"
            output["genbankCdsFetchError"] = ""
            return output

        fetchErrors.append(
            f"{proteinAccession}: {cdsResult['ncbiFetchStatus']} | {cdsResult['ncbiFetchError']}"
        )

    output["genbankCdsFetchStatus"] = "failed_all_protein_crossrefs"
    output["genbankCdsFetchError"] = " || ".join(fetchErrors[:5])

    return output


# -----------------------------------------------
# Choose UniProt IDs from reactionDF_wBestRCMFP_wUniProtMetadata
# -----------------------------------------------
reactionDF_wBestRCMFP_wUniProtMetadata["uniprotAccession"] = (
    reactionDF_wBestRCMFP_wUniProtMetadata["uniprotAccession"]
    .apply(cleanUniProtAccession)
)

reactionDF_wBestRCMFP_wUniProtMetadata["isValidUniProtAccession"] = (
    reactionDF_wBestRCMFP_wUniProtMetadata["uniprotAccession"]
    .apply(isValidUniProtAccession)
)

uniqueUniProtForGenBankList = sorted(
    reactionDF_wBestRCMFP_wUniProtMetadata.loc[
        reactionDF_wBestRCMFP_wUniProtMetadata["isValidUniProtAccession"],
        "uniprotAccession",
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

if maxUniProtToFetch is not None:
    uniqueUniProtForGenBankList = uniqueUniProtForGenBankList[:maxUniProtToFetch]

print(f"Unique UniProt accessions selected for GenBank CDS fetch: {len(uniqueUniProtForGenBankList):,}")

if len(uniqueUniProtForGenBankList) == 0:
    raise RuntimeError("No valid UniProt accessions found for GenBank CDS fetch.")


# -----------------------------------------------
# Fetch GenBank CDS sequences
# -----------------------------------------------
genbankLookupRecords = []

for uniprotAccession in tqdm(
    uniqueUniProtForGenBankList,
    desc="Fetching GenBank CDS via UniProt crossrefs",
):
    record = fetchBestGenBankCdsForUniProt(uniprotAccession)
    genbankLookupRecords.append(record)

genbankLookupDF = pd.DataFrame(genbankLookupRecords)

print("\nGenBank CDS fetch status:")
print(genbankLookupDF["genbankCdsFetchStatus"].value_counts(dropna=False))


# -----------------------------------------------
# Merge GenBank CDS back to reaction dataframe
# -----------------------------------------------
reactionDF_wBestRCMFP_wDNAseq = reactionDF_wBestRCMFP_wUniProtMetadata.merge(
    genbankLookupDF,
    on="uniprotAccession",
    how="left",
)

reactionDF_wBestRCMFP_wDNAseq["hasGenBankCdsSequence"] = (
    reactionDF_wBestRCMFP_wDNAseq["genbankCdsSequence"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

print("\nreactionDF_wBestRCMFP_wDNAseq created")
print(f"Rows total              : {len(reactionDF_wBestRCMFP_wDNAseq):,}")
print(f"Rows with GenBank CDS   : {reactionDF_wBestRCMFP_wDNAseq['hasGenBankCdsSequence'].sum():,}")

reactionDF_wBestRCMFP_wDNAseq.head()

Unique UniProt accessions selected for GenBank CDS fetch: 35


Fetching GenBank CDS via UniProt crossrefs: 100%|██████████████████████████████████████████████████████████| 35/35 [01:07<00:00,  1.93s/it]


GenBank CDS fetch status:
genbankCdsFetchStatus
success                33
no_protein_crossref     2
Name: count, dtype: int64

reactionDF_wBestRCMFP_wDNAseq created
Rows total              : 4,733
Rows with GenBank CDS   : 3,477


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,uniprotCrossRefStatus,uniprotCrossRefError,genbankProteinAccessionUsed,genbankCdsFastaHeader,genbankCdsSequence,genbankCdsLengthBp,numFastaRecordsReturned,genbankCdsFetchStatus,genbankCdsFetchError,hasGenBankCdsSequence
0,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,No_Thermo,"(1, 1, 1, 1)","(1, 1, 1)",Enzymatic,4,3,...,success,,ABO67118.1,lcl|CP000557.1_cds_ABO67118.1_1 [locus_tag=GTN...,ATGAGTGTAGCCCGCATTGTCTTTCCGCCGCTCAGCCATGTCGGCT...,1188.0,1.0,success,,True
1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,success,,XP_019041020.1,lcl|XM_019181327.1_cds_XP_019041020.1_1 [locus...,ATGTTTTTCACAAAAGTACTAAATAACCAAGTTGCCAATGGTTTAA...,1176.0,1.0,success,,True
3,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,success,,XP_019041020.1,lcl|XM_019181327.1_cds_XP_019041020.1_1 [locus...,ATGTTTTTCACAAAAGTACTAAATAACCAAGTTGCCAATGGTTTAA...,1176.0,1.0,success,,True
4,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,success,,XP_019041020.1,lcl|XM_019181327.1_cds_XP_019041020.1_1 [locus...,ATGTTTTTCACAAAAGTACTAAATAACCAAGTTGCCAATGGTTTAA...,1176.0,1.0,success,,True


### Remove those coumns if no DNA sequence found from `GenBank`

In [56]:
before_rows = len(reactionDF_wBestRCMFP_wDNAseq)

reactionDF_wBestRCMFP_wDNAseq = reactionDF_wBestRCMFP_wDNAseq[reactionDF_wBestRCMFP_wDNAseq["hasGenBankCdsSequence"] != False].copy()
after_rows = len(reactionDF_wBestRCMFP_wDNAseq)
dropped_rows = before_rows - after_rows

print(f"Total reaction : {before_rows:,}")
print(f"Reaction with GenBank CDS  : {after_rows:,}")
print(f"Reaction dropped : {dropped_rows:,}")

Total reaction : 4,733
Reaction with GenBank CDS  : 3,477
Reaction dropped : 1,256


## 7. Codon-optimize selected enzymes for `target_species` using `DNA Chisel`

### Print Organisms details: `E.coli/Pseudomonas putida` etc

In [57]:
organismCountsDF = (
    reactionDF_wBestRCMFP_wDNAseq["organism"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .value_counts(dropna=False)
    .rename_axis("organism")
    .reset_index(name="count")
)

totalRows = len(reactionDF_wBestRCMFP_wDNAseq)
organismCountsDF["percentage"] = (organismCountsDF["count"] / totalRows * 100).round(2)

print(f"Unique organisms: {organismCountsDF['organism'].nunique()}\n")
organismCountsDF

Unique organisms: 25



,organism,count,percentage
0,Homo sapiens (Human),789,22.69
1,Geobacillus thermodenitrificans (strain NG80-2),652,18.75
2,Aspergillus fumigatus (strain ATCC MYA-4609 / ...,553,15.90
3,Wickerhamomyces anomalus (strain ATCC 58044 / ...,330,9.49
4,Escherichia coli (strain K12),295,8.48
5,Comamonas testosteroni (Pseudomonas testosteroni),238,6.84
6,Taxus cuspidata (Japanese yew),153,4.40
7,Bluetongue virus (BTV),130,3.74
8,Severe acute respiratory syndrome coronavirus ...,105,3.02
9,Plagiochasma appendiculatum,75,2.16


### Codon optimization settings

In [58]:
targetSpecies = "e_coli"
minGc = 0.30
maxGc = 0.70
gcWindow = 50
stopCodon = "TAA"

forbiddenPatternList = [
    "BsaI_site",
    "BsmBI_site",
    "EcoRI_site",
    "XbaI_site",
    "SpeI_site",
    "PstI_site",
    "NotI_site",
]

stopCodons = {"TAA", "TAG", "TGA"}

In [59]:
# -----------------------------------------------
# Helper functions
# -----------------------------------------------
def cleanDna(seq):
    if pd.isna(seq):
        return ""
    return re.sub(r"[^ACGTN]", "", str(seq).upper())


def removeTerminalStop(cds):
    cds = cleanDna(cds)
    if len(cds) >= 3 and len(cds) % 3 == 0 and cds[-3:] in stopCodons:
        return cds[:-3]
    return cds


def hasInternalStop(cds):
    cds = cleanDna(cds)
    if len(cds) % 3 != 0:
        return True
    codons = [cds[i:i+3] for i in range(0, len(cds), 3)]
    return any(codon in stopCodons for codon in codons)


def classifyCdsForOptimization(cds):
    cds = cleanDna(cds)
    cdsNoStop = removeTerminalStop(cds)

    if len(cds) == 0:
        return "no_cds"
    if len(cdsNoStop) == 0:
        return "empty_after_stop_removal"
    if len(cdsNoStop) % 3 != 0:
        return "length_not_multiple_of_3"
    if "N" in cdsNoStop:
        return "contains_N"
    if hasInternalStop(cdsNoStop):
        return "contains_internal_stop"

    return "ready_for_optimization"


def safeText(x):
    if pd.isna(x):
        return ""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))[:80]


def wrapFasta(seq, width=80):
    seq = str(seq)
    return "\n".join(seq[i:i+width] for i in range(0, len(seq), width))


def optimizeCdsForEcoli(initialCds):
    sequenceLength = len(initialCds)
    geneLocation = (0, sequenceLength)

    constraints = [
        EnforceTranslation(location=geneLocation),
        EnforceGCContent(mini=minGc, maxi=maxGc, window=gcWindow),
    ]

    for patternName in forbiddenPatternList:
        constraints.append(AvoidPattern(patternName))

    objectives = [
        MaximizeCAI(species=targetSpecies, location=geneLocation)
    ]

    optimizationProblem = DnaOptimizationProblem(
        sequence=initialCds,
        constraints=constraints,
        objectives=objectives,
    )

    optimizationProblem.resolve_constraints()
    optimizationProblem.optimize()

    return optimizationProblem.sequence, optimizationProblem


# -----------------------------------------------
# Prepare input from reactionDF_wBestRCMFP_wDNAseq
# -----------------------------------------------
reactionDF_wBestRCMFP_wDNAseq_forOpt = reactionDF_wBestRCMFP_wDNAseq.copy()

reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequence"] = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequence"]
    .apply(cleanDna)
)

reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequenceForOptimization"] = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequence"]
    .apply(removeTerminalStop)
)

reactionDF_wBestRCMFP_wDNAseq_forOpt["codonOptimizationInputStatus"] = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequence"]
    .apply(classifyCdsForOptimization)
)

reactionDF_wBestRCMFP_wDNAseq_forOpt["isReadyForCodonOptimization"] = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt["codonOptimizationInputStatus"]
    == "ready_for_optimization"
)

reactionDF_wBestRCMFP_wDNAseq_forOpt["optimizationInputKey"] = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt["uniprotAccession"].fillna("").astype(str)
    + "|"
    + reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankProteinAccessionUsed"].fillna("").astype(str)
    + "|"
    + reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequenceForOptimization"].fillna("").astype(str)
).apply(lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest())

print("Input status:")
print(reactionDF_wBestRCMFP_wDNAseq_forOpt["codonOptimizationInputStatus"].value_counts(dropna=False))


# -----------------------------------------------
# Optimize unique GenBank CDS records only
# -----------------------------------------------
geneInputDF = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt[
        reactionDF_wBestRCMFP_wDNAseq_forOpt["isReadyForCodonOptimization"]
    ]
    .drop_duplicates("optimizationInputKey")
    .reset_index(drop=True)
)

optimizedGeneRecords = []

for idx, row in tqdm(
    geneInputDF.iterrows(),
    total=len(geneInputDF),
    desc="Optimizing RCMFP-supported GenBank CDS for E. coli"
):
    optimizationInputKey = row["optimizationInputKey"]
    initialCds = row["genbankCdsSequenceForOptimization"]

    try:
        optimizedCdsNoStop, optimizationProblem = optimizeCdsForEcoli(initialCds)
        optimizedCds = optimizedCdsNoStop + stopCodon

        optimizationStatus = "success"
        optimizationError = ""
        constraintsSummary = optimizationProblem.constraints_text_summary()
        objectivesSummary = optimizationProblem.objectives_text_summary()

    except Exception as exc:
        optimizedCds = ""
        optimizationStatus = "failed"
        optimizationError = str(exc)
        constraintsSummary = ""
        objectivesSummary = ""

    optimizedGeneRecords.append({
        "optimizationInputKey": optimizationInputKey,
        "optimizedGeneId": f"ecoli_rcmfp_opt_gene_{idx:06d}",
        "optimizedCds": optimizedCds,
        "optimizedCdsLengthBp": len(optimizedCds),
        "expressionHost": "Escherichia coli",
        "targetSpecies": targetSpecies,
        "minGc": minGc,
        "maxGc": maxGc,
        "gcWindow": gcWindow,
        "stopCodonUsed": stopCodon,
        "codonOptimizationStatus": optimizationStatus,
        "codonOptimizationError": optimizationError,
        "constraintsSummary": constraintsSummary,
        "objectivesSummary": objectivesSummary,
    })

optimizedGeneDF = pd.DataFrame(optimizedGeneRecords)

print("\nOptimization status:")
print(optimizedGeneDF["codonOptimizationStatus"].value_counts(dropna=False))


# -----------------------------------------------
# Merge optimized CDS back to reaction dataframe
# -----------------------------------------------
reactionDF_wOptimizedDNAseq = reactionDF_wBestRCMFP_wDNAseq_forOpt.merge(
    optimizedGeneDF,
    on="optimizationInputKey",
    how="left"
)

reactionDF_wOptimizedDNAseq["hasOptimizedCds"] = (
    reactionDF_wOptimizedDNAseq["optimizedCds"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

print("\nFinal reactionDF_wOptimizedDNAseq:")
print(f"Rows total           : {len(reactionDF_wOptimizedDNAseq):,}")
print(f"Rows optimized       : {reactionDF_wOptimizedDNAseq['hasOptimizedCds'].sum():,}")
print(f"Unique optimized CDS : {reactionDF_wOptimizedDNAseq.loc[reactionDF_wOptimizedDNAseq['hasOptimizedCds'], 'optimizationInputKey'].nunique():,}")
reactionDF_wOptimizedDNAseq.head()

Input status:
codonOptimizationInputStatus
ready_for_optimization    3477
Name: count, dtype: int64


Optimizing RCMFP-supported GenBank CDS for E. coli:   0%|                                                           | 0/33 [00:00<?, ?it/s]
constraint:   0%|                                                                  | 0/8 [00:00<?, ?it/s, now=AvoidPattern[0-1185](patt...]
                                                                                                                                           
objective:   0%|                                                                   | 0/1 [00:00<?, ?it/s, now=MaximizeCAI[0-1185](e_col...]

location:   0%|                                                                                          | 0/216 [00:00<?, ?it/s, now=None]

location:   0%|                                                                                           | 0/216 [00:00<?, ?it/s, now=3-6]

                                                                                                                                           
Optimizing RCMFP-


Optimization status:
codonOptimizationStatus
success    33
Name: count, dtype: int64

Final reactionDF_wOptimizedDNAseq:
Rows total           : 3,477
Rows optimized       : 3,477
Unique optimized CDS : 33


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,targetSpecies,minGc,maxGc,gcWindow,stopCodonUsed,codonOptimizationStatus,codonOptimizationError,constraintsSummary,objectivesSummary,hasOptimizedCds
0,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,rule0165_2,No_Thermo,"(1, 1, 1, 1)","(1, 1, 1)",Enzymatic,4,3,...,e_coli,0.3,0.7,50,TAA,success,,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -12.57\n -1...,True
1,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,e_coli,0.3,0.7,50,TAA,success,,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -7.88\n -...,True
2,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,e_coli,0.3,0.7,50,TAA,success,,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -7.88\n -...,True
3,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,e_coli,0.3,0.7,50,TAA,success,,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -7.88\n -...,True
4,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,rule0061_4,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,e_coli,0.3,0.7,50,TAA,success,,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -7.88\n -...,True


## 8. Export `DNA design` files for `Teselagen`

In [60]:
optimizedDNAFilePath = os.path.join(DNADesignResultsDir,"reactionDF_wBestRCMFP_optimized_Ecoli.fasta")

fastaDF = (reactionDF_wOptimizedDNAseq[reactionDF_wOptimizedDNAseq["hasOptimizedCds"]].drop_duplicates("optimizationInputKey").copy())

with open(optimizedDNAFilePath, "w", encoding="utf-8") as f:
    for _, row in fastaDF.iterrows():
        fastaHeader = (
            f">{row['optimizedGeneId']}|"
            f"rule={safeText(row.get('ruleName', ''))}|"
            f"UniProt={safeText(row.get('uniprotAccession', ''))}|"
            f"GenBankProtein={safeText(row.get('genbankProteinAccessionUsed', ''))}|"
            f"EC={safeText(row.get('ecNumber', ''))}|"
            f"host=Escherichia_coli"
        )

        f.write(fastaHeader + "\n")
        f.write(wrapFasta(row["optimizedCds"]) + "\n\n")

print(f"\nSaved FASTA at: {optimizedDNAFilePath}")


Saved FASTA at: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/reactionDF_wBestRCMFP_optimized_Ecoli.fasta


### Find unique `starter --> target` from reconstructed pathways

In [61]:
# Count complete starter --> target reconstructed pathways per directory
to_bool = lambda s: s.astype(str).str.lower().isin(["true", "1", "yes"])

first = reactionDF_wOptimizedDNAseq[reactionDF_wOptimizedDNAseq["step"].eq(1)]
final = reactionDF_wOptimizedDNAseq[reactionDF_wOptimizedDNAseq["step"].eq(reactionDF_wOptimizedDNAseq["numSteps"])]

complete_ids = set(first.loc[to_bool(first["starterInReactants"]), "routeId"].astype(str)) & \
               set(final.loc[to_bool(final["targetInProducts"]), "routeId"].astype(str))

completeRouteLevelDF = (
    reactionDF_wOptimizedDNAseq[reactionDF_wOptimizedDNAseq["routeId"].astype(str).isin(complete_ids)]
    .sort_values(["dirName", "routeId", "step"])
    .groupby("routeId", as_index=False)
    .agg(
        dirName=("dirName", "first"),
        jobName=("jobName", "first"),
        sourceFolderNum=("sourceFolderNum", "first"),
        numSteps=("numSteps", "first"),
        searchDepthUsed=("searchDepthUsed", "first"),
        starterSMILES=("starterSMILES", "first"),
        targetSMILES=("targetSMILES", "first"),
        reconstructedPathwayString=("reconstructedPathwayString", "first"),
    )
)

starterTargetPathwayCountsDF = (
    completeRouteLevelDF
    .pivot_table(
        index=["dirName", "jobName", "starterSMILES", "targetSMILES"],
        columns="numSteps",
        values="routeId",
        aggfunc="nunique",
        fill_value=0,
    )
    .rename(columns={1: "numOneStepPathways", 2: "numTwoStepPathways", 3: "numThreeStepPathways"})
    .reset_index()
)

for c in ["numOneStepPathways", "numTwoStepPathways", "numThreeStepPathways"]:
    if c not in starterTargetPathwayCountsDF.columns:
        starterTargetPathwayCountsDF[c] = 0

starterTargetPathwayCountsDF["totalStarterToTargetPathways"] = (
    starterTargetPathwayCountsDF["numOneStepPathways"] +
    starterTargetPathwayCountsDF["numTwoStepPathways"] +
    starterTargetPathwayCountsDF["numThreeStepPathways"]
)

starterTargetPathwayCountsDF = starterTargetPathwayCountsDF.sort_values(
    "totalStarterToTargetPathways", ascending=False
).reset_index(drop=True)

print(f"Total reaction pathways : {len(reactionDF_wOptimizedDNAseq):,}")
print(f"Complete starter --> target pathways: {len(completeRouteLevelDF):,}")
print(f"Directories reached starter --> target pathways: {starterTargetPathwayCountsDF['dirName'].nunique():,}")

starterTargetPathwayCountsDF

Total reaction pathways : 3,477
Complete starter --> target pathways: 233
Directories reached starter --> target pathways: 4


numSteps,dirName,jobName,starterSMILES,targetSMILES,numOneStepPathways,numTwoStepPathways,numThreeStepPathways,totalStarterToTargetPathways
0,high_pPotency_molecule_pathway2,high_pPotency_molecule_pathway2,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(C)(COP(=O)(O)OP(=O)(O)OCC1OC(n2cnc3c(N)ncnc...,1,7,112,120
1,high_pPotency_molecule_pathway6,high_pPotency_molecule_pathway6,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,1,0,58,59
2,high_pPotency_molecule_pathway1,high_pPotency_molecule_pathway1,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP...,0,0,50,50
3,high_pPotency_molecule_pathway18,high_pPotency_molecule_pathway18,O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O,NC(=O)c1ccc[n+](C2OC(COP(=O)(O)OP(=O)(O)OCC3OC...,0,1,3,4
